# Transformacoes Gold - Siplan RPS

Le as tabelas raw de `lake_prep_siplan` (populadas pelo Dataflow Gen2) e salva as tabelas de producao em `lake_gold_siplan`.

```
lake_prep_siplan   (raw_acoes, raw_tags, raw_datas, raw_projetos,
                    raw_datas_sessoes, raw_acessibilidade,
                    raw_solicitacoes, raw_pcap, raw_pcap_detalhe,
                    raw_parcelas, raw_acoes_txts, raw_pcac_det)
    v  Fabric Notebook (este)
lake_gold_siplan   (tabela_base, datas_sessoes, contratos, pcap)
```

todas_as_datas e datas_sesseoes com problemas
e todas_as_tags?


## Config do ambiente

In [1]:
import re
import warnings
import numpy as np
import pandas as pd

MONTH_ABBR = {
    1: 'jan', 2: 'fev', 3: 'mar', 4: 'abr', 5: 'mai', 6: 'jun',
    7: 'jul', 8: 'ago', 9: 'set', 10: 'out', 11: 'nov', 12: 'dez',
}

WEEKDAY_ABBR = {0: 'seg', 1: 'ter', 2: 'qua', 3: 'qui',
                4: 'sex', 5: 'sab', 6: 'dom'}

# Datas anteriores a 1900 gravadas por versões antigas do Spark/Hive usam
# calendário Julian; "CORRECTED" trata como Gregoriano sem tentar rebasear.
# Valores serão descartados depois pelo errors='coerce' em pd.to_datetime.
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead",  "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead",     "CORRECTED")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite",    "CORRECTED")

def read_raw(table_name: str) -> pd.DataFrame:
    return spark.sql(f'SELECT * FROM lake_prep_siplan.dbo.{table_name}').toPandas()


def read_raw_filtered(table_name: str, ids: pd.Series, dedup_col: str = None, src_col: str = 'atividade_id') -> pd.DataFrame:
    """Lê tabela filtrada por atividade_id via join no Spark.

    dedup_col: se informado, mantém apenas uma linha por esse valor (ROW_NUMBER=1).
    Use para tabelas 1:1 cujo Dataflow acumula cópias a cada refresh (_all mode).
    """
    (
        spark.createDataFrame(ids.drop_duplicates().to_frame())
        .createOrReplaceTempView('_filter_ids')
    )

    _sc = f'`{src_col}`' if '.' in src_col else src_col

    if dedup_col:
        sql = (
            f'SELECT t.* FROM ('
            f'  SELECT /*+ BROADCAST(f) */ t2.*,'
            f'    ROW_NUMBER() OVER (PARTITION BY t2.{dedup_col} ORDER BY t2.{dedup_col}) AS _rn'
            f'  FROM lake_prep_siplan.dbo.{table_name} t2'
            f'  INNER JOIN _filter_ids f ON t2.{_sc} = f.atividade_id'
            f') t WHERE t._rn = 1'
        )
        df = spark.sql(sql).toPandas()
        return df.drop(columns=['_rn'], errors='ignore')

    return spark.sql(
        f'SELECT /*+ BROADCAST(f) */ t.* '
        f'FROM lake_prep_siplan.dbo.{table_name} t '
        f'INNER JOIN _filter_ids f ON t.{_sc} = f.atividade_id'
    ).toPandas()

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 3, Finished, Available, Finished, False)

## 1. Carregar tabelas raw


In [2]:
HISTORICO = False   # True = _all (histórico completo);  False = só ano corrente
_all = "_all" if HISTORICO else ""

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 4, Finished, Available, Finished, False)

In [3]:
# Ações, Projetos, Tags e Acessibilidade

raw_acoes_df = read_raw(f'raw_acoes{_all}')
raw_acoes_df['atividade_id'] = raw_acoes_df['atividade_id'].astype(str).str.strip()
raw_acoes_df['servico']      = raw_acoes_df['servico'].astype(str).str.strip()
raw_acoes_df['subatividade'] = raw_acoes_df['subatividade'].astype(str).str.strip()
raw_acoes_df = raw_acoes_df.drop_duplicates(subset=['atividade_id'])
# Normaliza projeto_id (pode vir com prefixo 'a.' do alias SQL)
_acoes_proj_col = next((c for c in raw_acoes_df.columns if c == 'projeto_id' or c.endswith('.projeto_id')), None)
if _acoes_proj_col and _acoes_proj_col != 'projeto_id':
    raw_acoes_df = raw_acoes_df.rename(columns={_acoes_proj_col: 'projeto_id'})
if 'projeto_id' in raw_acoes_df.columns:
    raw_acoes_df['projeto_id'] = raw_acoes_df['projeto_id'].astype(str).str.strip()
# Normaliza uo (pode vir como a.uo do alias SQL)
_acoes_uo_col = next((c for c in raw_acoes_df.columns if c == 'uo' or c.endswith('.uo')), None)
if _acoes_uo_col and _acoes_uo_col != 'uo':
    raw_acoes_df = raw_acoes_df.rename(columns={_acoes_uo_col: 'uo'})
print(f'raw_acoes_df:        {raw_acoes_df.shape}')

raw_tags_df = read_raw(f'raw_tags{_all}')
raw_tags_df['atividade_id'] = raw_tags_df['atividade_id'].astype(str).str.strip()
print(f'raw_tags_df:         {raw_tags_df.shape}')

raw_acessibilidade_df = read_raw_filtered(f'raw_acessibilidade{_all}', raw_acoes_df['atividade_id'])
raw_acessibilidade_df['atividade_id'] = raw_acessibilidade_df['atividade_id'].astype(str).str.strip()
print(f'raw_acessibilidade:  {raw_acessibilidade_df.shape}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 5, Finished, Available, Finished, False)

raw_acoes_df:        (35417, 27)
raw_tags_df:         (39528, 4)
raw_acessibilidade:  (2328, 4)


## Diagnóstico: Campos de Público

Células D1–D4: qualidade de `a.lugares` e `a.estimativa_publico`,
diversidade de locais por atividade e cobertura de capacidade por serviço.

> **D1** roda imediatamente após o carregamento de `raw_acoes_df`.
> **D2–D4** requerem `raw_datas_sessoes_df` — executar após a seção de datas.

In [ ]:
# ── Diagnóstico D1: campos de público em raw_acoes_df ──────────────────
# Filtra apenas atividades com status relevante (PENDENTE / APROVADO)
_status_col = 'status_atividade' if 'status_atividade' in raw_acoes_df.columns else 'a.status_atividade'
_mask_status = raw_acoes_df[_status_col].str.upper().isin(['PENDENTE', 'APROVADO'])
_ids_validos = set(raw_acoes_df.loc[_mask_status, 'atividade_id'])
print(f'Atividades PENDENTE/APROVADO: {len(_ids_validos)} de {len(raw_acoes_df)} ({len(_ids_validos)/len(raw_acoes_df):.1%})\n')

_pub = raw_acoes_df[_mask_status][['atividade_id', 'a.lugares', 'a.estimativa_publico',
                                   'servico', 'subatividade']].copy()
_pub['lugares_n']    = pd.to_numeric(_pub['a.lugares'],            errors='coerce')
_pub['estimativa_n'] = pd.to_numeric(_pub['a.estimativa_publico'], errors='coerce')

n = len(_pub)
print(f'Total atividades (filtradas): {n}\n')

print('─── Nulos / zeros ───')
print(f'  lugares nulo:         {_pub["lugares_n"].isna().sum():>6}  ({_pub["lugares_n"].isna().mean():.1%})')
print(f'  lugares = 0:          {(_pub["lugares_n"] == 0).sum():>6}  ({(_pub["lugares_n"] == 0).mean():.1%})')
print(f'  estimativa nula:      {_pub["estimativa_n"].isna().sum():>6}  ({_pub["estimativa_n"].isna().mean():.1%})')
print(f'  estimativa = 0:       {(_pub["estimativa_n"] == 0).sum():>6}  ({(_pub["estimativa_n"] == 0).mean():.1%})')
_ambos_nulos = _pub['lugares_n'].isna() & _pub['estimativa_n'].isna()
print(f'  AMBOS nulos (→ 1):    {_ambos_nulos.sum():>6}  ({_ambos_nulos.mean():.1%})')

print('\n─── Distribuição de lugares (onde preenchido e > 0) ───')
_lug_ok = _pub['lugares_n'].dropna()
_lug_ok = _lug_ok[_lug_ok > 0]
print(_lug_ok.describe(percentiles=[.25,.5,.75,.90,.95,.99]).round(0))
print(f'\n  > 4000:  {(_lug_ok > 4000).sum()} atividades')
print(f'  > 1000:  {(_lug_ok > 1000).sum()} atividades')

print('\n─── Distribuição de estimativa_publico (onde preenchido e > 0) ───')
_est_ok = _pub['estimativa_n'].dropna()
_est_ok = _est_ok[_est_ok > 0]
print(_est_ok.describe(percentiles=[.25,.5,.75,.90,.95,.99]).round(0))
print(f'\n  > 4000:  {(_est_ok > 4000).sum()} atividades')

print('\n─── Ambos nulos por serviço (top 15) ───')
print(
    _pub[_ambos_nulos]
    .groupby('servico', as_index=False)
    .size()
    .sort_values('size', ascending=False)
    .head(15)
    .to_string(index=False)
)

_ambos_ok = (
    _pub['lugares_n'].notna() & _pub['estimativa_n'].notna()
    & (_pub['lugares_n'] > 0) & (_pub['estimativa_n'] > 0)
)
_disc = _pub[_ambos_ok].copy()
_disc['razao'] = _disc['lugares_n'] / _disc['estimativa_n']
print(f'\n─── Discordância lugares vs. estimativa (ambos preenchidos: {_ambos_ok.sum()}) ───')
print(_disc['razao'].describe(percentiles=[.05,.25,.5,.75,.95]).round(2))
_fator5 = (_disc['razao'] > 5) | (_disc['razao'] < 0.2)
print(f'  Discordância > 5x:  {_fator5.sum()} atividades')

### Diagnóstico de duplicatas

In [4]:
# # Diagnóstico: verifica duplicatas por atividade_id em cada tabela _all
# # (lightweight — só COUNT, não coleta dados para o driver)

# for tbl in ['raw_acessibilidade', 'raw_datas', 'raw_datas_sessoes', 'raw_solicitacoes', 'raw_acoes_txts']:
#     spark.sql(f"""
#         SELECT
#             '{tbl}_all'           AS tabela,
#             COUNT(*)              AS total_linhas,
#             COUNT(DISTINCT atividade_id) AS atividades_unicas,
#             ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT atividade_id), 1) AS media_copias
#         FROM lake_prep_siplan.dbo.{tbl}_all
#     """).show(truncate=False)

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 6, Finished, Available, Finished, False)

## Projetos

In [5]:
# Projetos — grain: uma linha por projeto_id
_proj_ids = raw_acoes_df['projeto_id'].dropna().drop_duplicates()
spark.createDataFrame(_proj_ids.to_frame()).createOrReplaceTempView('_filter_proj_ids')

# Detecta nome da coluna no Delta (pode ter prefixo 'p.' do alias SQL)
_rp_cols   = spark.sql(f'SELECT * FROM lake_prep_siplan.dbo.raw_projetos{_all} LIMIT 0').columns
_rp_id_col = next((c for c in _rp_cols if c == 'projeto_id' or c.endswith('.projeto_id')), 'projeto_id')
_rp_id_ref = f'`{_rp_id_col}`' if '.' in _rp_id_col else _rp_id_col

raw_projetos_df = spark.sql(
    f'SELECT /*+ BROADCAST(f) */ t.* '
    f'FROM lake_prep_siplan.dbo.raw_projetos{_all} t '
    f'INNER JOIN _filter_proj_ids f ON t.{_rp_id_ref} = f.projeto_id'
).toPandas()
# Strip de qualquer prefixo de alias SQL (p., pu., etc.)
raw_projetos_df.columns = [c.split('.')[-1] for c in raw_projetos_df.columns]
raw_projetos_df['projeto_id'] = raw_projetos_df['projeto_id'].astype(str).str.strip()
print(f'raw_projetos_df:     {raw_projetos_df.shape}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 7, Finished, Available, Finished, False)

raw_projetos_df:     (1299, 14)


In [6]:
# Datas — 1:1 por atividade; dedup_col descarta cópias do Dataflow
raw_datas_df = read_raw_filtered(f'raw_datas{_all}', raw_acoes_df['atividade_id'], dedup_col='atividade_id')
raw_datas_df['atividade_id'] = raw_datas_df['atividade_id'].astype(str).str.strip()
print(f'raw_datas_df:        {raw_datas_df.shape}')

# datas_sessoes — várias linhas por atividade (uma por sessão); sem dedup
raw_datas_sessoes_raw_df = read_raw_filtered(f'raw_datas_sessoes{_all}', raw_acoes_df['atividade_id'])
print(f'raw_datas_sessoes:   {raw_datas_sessoes_raw_df.shape}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 8, Finished, Available, Finished, False)

raw_datas_df:        (35272, 20)
raw_datas_sessoes:   (580721, 8)


In [7]:
# Solicitações
raw_solicitacoes_df = read_raw_filtered(f'raw_solicitacoes{_all}', raw_acoes_df['atividade_id'])
raw_solicitacoes_df['atividade_id']   = raw_solicitacoes_df['atividade_id'].astype(str).str.strip()
raw_solicitacoes_df['solicitacao_id'] = pd.to_numeric(raw_solicitacoes_df['solicitacao_id'], errors='coerce').astype('Int64')
raw_solicitacoes_df['custo']          = pd.to_numeric(raw_solicitacoes_df['custo'], errors='coerce').fillna(0)
raw_solicitacoes_df = raw_solicitacoes_df[
    raw_solicitacoes_df['atividade_id'].isin(raw_acoes_df['atividade_id'])
].copy()
print(f'raw_solicitacoes:    {raw_solicitacoes_df.shape}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 9, Finished, Available, Finished, False)

raw_solicitacoes:    (51493, 9)


In [8]:
# PCAP
pcap_props_df = read_raw(f'raw_pcap{_all}')
pcap_props_df['pcap_num'] = pd.to_numeric(pcap_props_df['pcap_num'], errors='coerce').astype('Int64')
print(f'raw_pcap:            {pcap_props_df.shape}')

# Parcelas
raw_parcelas_df = read_raw('raw_parcelas')
raw_parcelas_df['solicitacao_id'] = pd.to_numeric(raw_parcelas_df['solicitacao_id'], errors='coerce').astype('Int64')
print(f'raw_parcelas_df:     {raw_parcelas_df.shape}')

# PCAC Detalhe
raw_pcac_det_df = read_raw('raw_pcap_det')
print(f'raw_pcap_det_df:     {raw_pcac_det_df.shape}')


StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 10, Finished, Available, Finished, False)

raw_pcap:            (16883, 22)
raw_parcelas_df:     (48585, 7)
raw_pcap_det_df:     (99138, 12)


In [9]:
# Acoes Textos - tem versao _all; filtrado por atividade_id
# coluna de join na fonte tem prefixo 'acao.' (Dataflow)
raw_acoes_txts_df = read_raw_filtered(f'raw_acoes_txts{_all}', raw_acoes_df['atividade_id'], src_col='acao.atividade_id')
raw_acoes_txts_df['acao.atividade_id'] = raw_acoes_txts_df['acao.atividade_id'].astype(str).str.strip()
print(f'raw_acoes_txts_df:   {raw_acoes_txts_df.shape}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 11, Finished, Available, Finished, False)

raw_acoes_txts_df:   (21649, 7)


## 2. Tags e RPS Parcial


In [10]:
# todas_as_tags (calculado em Python — não vem da staging)
todas_tags = (
    raw_tags_df.dropna(subset=['tag_nome'])
    .groupby('atividade_id')['tag_nome']
    .apply(lambda x: ' | '.join(sorted(x.unique())))
    .reset_index(name='todas_as_tags')
)
raw_tags_df = raw_tags_df.drop(columns=['todas_as_tags'], errors='ignore').merge(todas_tags, on='atividade_id', how='left')
print(f'raw_tags_df (com todas_as_tags): {raw_tags_df.shape}')

# rps_parcial_df
sts_df = (
    raw_tags_df[raw_tags_df['tag_nome'] == 'Avaliação STS'][['atividade_id']]
    .assign(tag='Avaliação STS')
    .drop_duplicates()
)
rps_parcial_df = (
    raw_acoes_df[['atividade_id', 'servico', 'subatividade']]
    .merge(sts_df[['atividade_id', 'tag']], on='atividade_id', how='left')
    .assign(tag=lambda d: d['tag'].fillna(''))
    .drop_duplicates(subset=['atividade_id'])
)
print(f'rps_parcial_df: {rps_parcial_df.shape}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 12, Finished, Available, Finished, False)

raw_tags_df (com todas_as_tags): (39528, 5)
rps_parcial_df: (35417, 4)


## 3. Precificação


In [ ]:
# 1d. Parser de precificacao_desc → gratuito, maior_valor, menor_valor
#
# Regras:
#   - Extrai todos os valores "R$ XX.YY" ou "R$ XX,YY" via regex
#   - Detecta "grátis" / "gratuito" / "aberto" no texto (case-insensitive)
#   - gratuito = 'Sim' se não há valor pago (>0); 'Não' caso contrário
#   - maior_valor = max dos valores pagos encontrados (None se tudo grátis ou sem info)
#   - menor_valor = 0 se "grátis/aberto" coexiste com valores pagos; min dos pagos caso contrário

import re

# Ponto ou vírgula como separador decimal: "R$ 10.00" ou "R$ 10,00"
_RE_VALOR  = re.compile(r'R\$\s*(\d+[.,]\d{2})', re.IGNORECASE)
# "grátis", "gratis", "gratuito" ou "aberto" → acesso livre
_RE_GRATIS = re.compile(r'gr[áa]tis|gratuito|aberto', re.IGNORECASE)

# -- alertas de complemento (passagem v5) -----------------------------------
import unicodedata as _ud

_SEP_PASS     = r'(?:\s+[xX]\s+|\s*[/\->]\s*)'
_RE_ROTA_COD  = re.compile(
    r'\b[A-Za-z]{2,4}\b' + _SEP_PASS + r'(?:\d+\s*)?\b[A-Za-z]{2,4}\b',
    re.IGNORECASE
)
_RE_ROTA_CIDAD = re.compile(
    r'[A-Za-z]{3,}' + _SEP_PASS + r'[A-Za-z]{2,}',
    re.IGNORECASE
)
_RE_CONCAT_PASS  = re.compile(r'\b[A-Z]{2,4}(?:[xX][A-Z]{2,4})+\b')
_RE_NUM_UNIT_PASS = re.compile(
    r'\b\d+\s*(?:pax|px|pessoa[s]?|aéreas?|passagem|passagens|bilhetes?|trechos?|voos?)\b',
    re.IGNORECASE
)
_RE_KW_PASS = re.compile(
    r'\bpax\b|\bpx\b|\bpassagem\b|\bpassagens\b|\baéreas?\b',
    re.IGNORECASE
)
# Hospedagem: '4pax x 8 diarias' ou '4 x 8 diarias'
_RE_HOSPED_DIARIAS = re.compile(
    r'\b\d+\s*(?:pax|pessoa[s]?|pes\.?)?\s*[xX×]\s*\d+\s*(?:diári[ao]s?|diarias?|noites?)\b',
    re.IGNORECASE
)
# PCAP -- sem grupo de captura (para str.contains)
_RE_PCAP_CHECK = re.compile(r'PCAP[^0-9]*(?:[0-9]{13})', re.IGNORECASE)


def _valid_passagem(text: str) -> bool:
    t = str(text).strip()
    if not t:
        return False
    tn = ''.join(c for c in _ud.normalize('NFD', t) if _ud.category(c) != 'Mn')
    return (
        bool(_RE_ROTA_COD.search(tn))     or
        bool(_RE_ROTA_CIDAD.search(tn))   or
        bool(_RE_CONCAT_PASS.search(t))   or
        bool(_RE_NUM_UNIT_PASS.search(t)) or
        bool(_RE_KW_PASS.search(t))
    )


_RE_HOSP_KW = re.compile(
    r'(?<![a-zA-Z])(?:'
    r'diarias?|diarios?'
    r'|noites?'
    r'|pax|px'
    r'|hospedage(?:m|ns)'
    r'|hospedes?'
    r'|singles?'
    r'|duplos?|dbls?|dpls?'
    r'|triplos?|tpls?'
    r'|doubles?'
    r'|suites?'
    r'|sgls?'
    r'|twn'
    r'|quartos?'
    r'|aptos?'
    r'|hotel'
    r'|pessoas?'
    r')',
    re.IGNORECASE
)
_RE_PLAN_VALID = re.compile(
    r'\bplanilha\s+(?:inserida|anexa|corrigida|atualizada)',
    re.IGNORECASE
)


def _valid_hospedagem(text: str) -> bool:
    t = str(text).strip()
    if not t:
        return False
    tn = ''.join(c for c in _ud.normalize('NFD', t) if _ud.category(c) != 'Mn')
    return bool(_RE_HOSP_KW.search(tn)) or bool(_RE_PLAN_VALID.search(tn))


def _str_to_float(s: str) -> float:
    # Normaliza separador decimal: "10,00" → "10.00", "10.00" → "10.00"
    return float(s.replace(',', '.'))


def parse_precificacao(desc) -> tuple:
    """Retorna (gratuito, maior_valor, menor_valor) a partir de precificacao_desc."""
    if pd.isna(desc) or str(desc).strip() == '':
        return ('Não', None, None)

    s = str(desc)
    tem_gratis = bool(_RE_GRATIS.search(s))

    raw_vals = _RE_VALOR.findall(s)
    valores = []
    for v in raw_vals:
        try:
            valores.append(_str_to_float(v))
        except ValueError:
            pass

    pagos = [v for v in valores if v > 0]

    if not pagos and not tem_gratis:
        return ('Não', None, None)

    if not pagos:
        # Apenas grátis / aberto
        return ('Sim', 0.0, 0.0)

    # Há valores pagos
    maior = max(pagos)
    # "grátis/aberto" coexiste com pago → opção gratuita existe → menor = 0
    menor = 0.0 if tem_gratis else min(pagos)
    return ('Não', maior, menor)


# ── Aplica e exibe diagnóstico ────────────────────────────────────────────────
_res = raw_acoes_df['a.precificacao_desc'].apply(parse_precificacao)
precif_df = pd.DataFrame(_res.tolist(), columns=['gratuito', 'maior_valor', 'menor_valor'],
                         index=raw_acoes_df.index)
precif_df.insert(0, 'atividade_id', raw_acoes_df['atividade_id'])

print("gratuito:")
print(precif_df['gratuito'].value_counts())
print()
print("maior_valor — estatísticas:")
print(precif_df['maior_valor'].describe())
print()
print("menor_valor — estatísticas:")
print(precif_df['menor_valor'].describe())
print()

# Amostra de casos mistos (grátis/aberto + pago na mesma atividade)
_mistos = precif_df[(precif_df['gratuito'] == 'Não') & (precif_df['menor_valor'] == 0)]
print(f"Casos mistos (pago + grátis/aberto): {len(_mistos)}")
if len(_mistos) > 0:
    _check = raw_acoes_df.loc[_mistos.index, ['atividade_id', 'a.precificacao_desc']].head(5)
    for _, row in _check.iterrows():
        print(f"  {row['atividade_id']}: {str(row['a.precificacao_desc'])[:90]}")

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 13, Finished, Available, Finished, False)

gratuito:
gratuito
Não    18255
Sim    17162
Name: count, dtype: int64

maior_valor — estatísticas:
count    17162.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: maior_valor, dtype: float64

menor_valor — estatísticas:
count    17162.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: menor_valor, dtype: float64

Casos mistos (pago + grátis/aberto): 0


## 4. Datas


In [12]:
def transform_datas(df, rps_df):
    df = df.copy()

    # Remove prefixo de nome de coluna que o driver ODBC pode incluir
    df.columns = [col.split('.')[-1] for col in df.columns]

    # Normaliza atividade_id para string (driver pode retornar float64)
    df['atividade_id'] = df['atividade_id'].astype(str).str.strip()

    # --- data/hora da primeira sessão ---
    df['primeiradata'] = pd.to_datetime(df['primeiradata'], errors='coerce')
    df['PrimeiraData']     = df['primeiradata'].dt.date
    df['PrimeiraHora']     = df['primeiradata'].dt.time
    df['PrimeiraDataHora'] = df['primeiradata'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df['mes'] = df['primeiradata'].dt.month.map(MONTH_ABBR)

    # --- tempo médio da sessão (truncado a 1 decimal) ---
    df['tempo_da_sessao'] = (
        np.floor(pd.to_numeric(df['tempo_sessao'], errors='coerce').fillna(0) * 10) / 10
    )

    # --- flags de prazo (binário → 'sim'/'0') ---
    # 30dias: nº de datas distintas com sessão > 30  (≠ diascorridos > 30)
    # 90dias: diascorridos > 90
    # 60horas: total de horas > 60
    # Todos já calculados no SQL; apenas mapeamos para 'sim'/'0'
    bool_map = {1: 'sim', '1': 'sim', 0: '0', '0': '0'}
    for col in ['30dias', '90dias', '60horas']:
        df[col] = pd.to_numeric(df[col], errors='coerce').map(bool_map).fillna('0')

    df['ExtrapolaDataHora'] = np.where(
        (df['90dias'] == 'sim') | (df['60horas'] == 'sim'), 'sim', '0'
    )

    # --- autonomia temporal bruta ---
    # DIREG: ultrapassa 90 dias corridos OU 60 horas totais
    # STS  : mais de 30 datas distintas com sessão
    # UO   : demais casos
    conditions = [
        (df['90dias'] == 'sim') | (df['60horas'] == 'sim'),
        df['30dias'] == 'sim',
    ]
    df['autonomiaTemporal'] = np.select(conditions, ['DIREG', 'STS'], default='UO')

    # --- ajuste por serviço (via RPS Parcial) ---
    # A autonomia temporal só faz sentido para serviços com acúmulo de carga/dias.
    # Para os demais, UO é sempre o nível correto independente das datas.
    df = df.merge(rps_df[['atividade_id', 'servico', 'subatividade']], on='atividade_id', how='left')
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    usa_temporal = (
        df['servico'].isin({'Curso', 'Ioga'}) |
        df['servico'].str.startswith('Desenvolvimento') |
        df['subatividade'].str.startswith('Ações')
    )
    df['autonomiaTemporal'] = np.where(usa_temporal, df['autonomiaTemporal'], 'UO')

    # servico/subatividade já estarão na tabela base; remove daqui para evitar duplicatas
    df = df.drop(columns=['servico', 'subatividade'])
    return df

datas_df = transform_datas(raw_datas_df, rps_parcial_df)
print(f'datas_df: {datas_df.shape}')


StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 14, Finished, Available, Finished, False)

datas_df: (35272, 27)


## 5. Datas / Sessões


In [13]:
# Nota: diaSemama preserva o typo original do Power Query (era "semana").
# Renomear quebraria relatórios existentes que referenciem essa coluna.
WEEKDAY_ABBR = {0: "seg", 1: "ter", 2: "qua", 3: "qui",
                4: "sex", 5: "sab", 6: "dom"}

def transform_datas_sessoes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [col.split(".")[-1] for col in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]

    g = df['local_grupo']

    # Tipos base
    df["atividade_id"]  = df["atividade_id"].astype(str).str.strip()
    df["sessao_id"]     = df["sessao_id"].astype(str).str.strip()
    df["datainicio"]    = pd.to_datetime(df["datainicio"],    errors="coerce")
    df["datafinal"]     = pd.to_datetime(df["datafinal"],     errors="coerce")
    df["primeira_data"] = pd.to_datetime(df["primeira_data"], errors="coerce")

    # Remove sessões com datas fora do intervalo datetime64[ns] (ex: anos > 2262)
    n_before = len(df)
    df = df[df["datainicio"].notna()].copy()
    n_dropped = n_before - len(df)
    if n_dropped:
        print(f"[AVISO] {n_dropped} sessão(ões) descartada(s): datainicio inválida ou fora dos limites")

    # Exclui grupo "37" (tipologia sem validade)
    df = df[df["local_grupo"] != "37"].copy()

    # Duração e diferença em horas inteiras
    df["Duracao"]   = df["datafinal"] - df["datainicio"]
    df["diferenca"] = (df["Duracao"].dt.total_seconds() / 3600).astype(int)

    # Colunas de tempo derivadas de datainicio
    dt = df["datainicio"].dt
    df["Data"]         = dt.normalize()                        # datetime na meia-noite
    df["HoraCheia"]    = df["datainicio"].dt.floor("h").dt.time  # hora cheia (sem minutos)
    df["horaCerta"]    = dt.time                               # hora exata
    df["horaCertaTxt"] = dt.strftime("%H:%M")

    # PeriodoDia baseado na hora cheia
    hora = dt.hour
    df["PeriodoDia"] = np.select(
        [hora < 5, hora < 12, hora < 18],
        ["Madrugada", "de manhã", "à tarde"],
        default="à noite"
    )

    # TipoLocal — vetorizado (máscara de maior prioridade aplicada por último)
    g = df["local_grupo"].fillna("")
    l = df["local_nome"].fillna("")
    df["TipoLocal"] = "na UO"
    df.loc[g == "Fora da Unidade",              "TipoLocal"] = "externa"
    df.loc[l.str.contains("Online",       na=False), "TipoLocal"] = "online"
    df.loc[l.str.contains("Sesc Digital", na=False), "TipoLocal"] = "online"
    df.loc[g.str.contains("Online",       na=False), "TipoLocal"] = "online"

    # Renomeia e formata texto
    df = df.rename(columns={"local_grupo": "TipologiaLocal", "local_nome": "localNome"})
    df["localNome"] = df["localNome"].str.title()

    # Partes de data (de datainicio)
    df["ano"]           = dt.year
    df["Mes"]           = dt.month
    df["mesTxt"]        = dt.month.map(MONTH_ABBR)
    df["dia"]           = dt.day
    df["diaSemama"]     = dt.dayofweek             # seg=0, dom=6
    df["diaSemanaTxt"]  = dt.dayofweek.map(WEEKDAY_ABBR)
    df["SemanaDoAno"] = dt.isocalendar().week.astype(int)

    # Partes de data (de primeira_data — mês da 1ª sessão da atividade)
    p = df["primeira_data"].dt
    df["mes1o"]    = p.month
    df["mes1oTxt"] = p.month.map(MONTH_ABBR)

    # Remove colunas não consumidas downstream
    df = df.drop(columns=["local_id", "grupo_id", "correcao_local",
                          "uo_local", "geac", "datafinal"], errors="ignore")

    col_order = [
        "atividade_id", "sessao_id", "uo",
        "datainicio", "Data", "diferenca", "Duracao",
        "HoraCheia", "horaCerta", "horaCertaTxt", "PeriodoDia",
        "localNome", "TipologiaLocal", "TipoLocal", "local_grupo",
        "ano", "Mes", "mesTxt", "dia", "diaSemama", "diaSemanaTxt", "SemanaDoAno",
        "primeira_data", "primeira_hora", "mes1o", "mes1oTxt",
    ]
    return df[[c for c in col_order if c in df.columns]]


StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 15, Finished, Available, Finished, False)

In [14]:
raw_datas_sessoes_df = transform_datas_sessoes(raw_datas_sessoes_raw_df)
raw_datas_sessoes_df = raw_datas_sessoes_df.merge(
    raw_acoes_df[['atividade_id', 'uo']],
    on='atividade_id', how='left'
)
print(f'raw_datas_sessoes_df: {raw_datas_sessoes_df.shape}')
print(raw_datas_sessoes_df['TipoLocal'].value_counts())


StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 16, Finished, Available, Finished, False)

raw_datas_sessoes_df: (580721, 24)
TipoLocal
na UO      570317
externa      9414
online        990
Name: count, dtype: int64


In [ ]:
# ── Diagnóstico D2: diversidade de locais por atividade ────────────────
# Aplica mesmo filtro de status definido em D1 (_ids_validos)
# Atividades em múltiplos locais ou com capacidade variável por formato
# (ginásio com futsal e show) são VÁLIDAS. Aqui apenas reportamos a variância.

_sess = raw_datas_sessoes_df[
    raw_datas_sessoes_df['atividade_id'].isin(_ids_validos)
][['atividade_id', 'localNome', 'TipologiaLocal', 'TipoLocal']].copy()

_n_locais = (
    _sess.groupby('atividade_id')
    .agg(
        n_locais_distintos=('localNome',      'nunique'),
        n_tipologias      =('TipologiaLocal', 'nunique'),
        n_tipos           =('TipoLocal',      'nunique'),
    )
    .reset_index()
)

print('─── n° de localNome distintos por atividade ───')
print(_n_locais['n_locais_distintos'].value_counts().sort_index().head(10))
print(f'\n  2+ locais distintos: {(_n_locais["n_locais_distintos"] > 1).sum()}')
print(f'  5+ locais distintos: {(_n_locais["n_locais_distintos"] > 4).sum()}')
print('  NOTA: múltiplos locais são válidos; indicam que um único a.lugares')
print('        pode não representar todas as sessões da atividade.')

print('\n─── Atividades que misturam TipoLocal ───')
print(_n_locais['n_tipos'].value_counts().sort_index())

print('\n─── TipologiaLocal — top 30 ───')
print(_sess['TipologiaLocal'].value_counts().head(30).to_string())

# Locais abertos: sem lotação física definível
_RE_ABERTO = r'praça|convivência|externo|fora|parque|jardim|calçada|pátio|varanda|átrio|hall|foyer|lobby'
_sess['eh_aberto'] = (
    _sess['localNome'].fillna('').str.lower().str.contains(_RE_ABERTO)
    | _sess['TipologiaLocal'].fillna('').str.lower().str.contains(_RE_ABERTO)
    | (_sess['TipoLocal'] == 'externa')
)
_ativ_abertos = _sess.groupby('atividade_id')['eh_aberto'].any().reset_index()
_ativ_abertos.columns = ['atividade_id', 'tem_sessao_aberta']
print(f'\n─── Atividades com ≥1 sessão em local aberto/externo ───')
print(f'  Total: {_ativ_abertos["tem_sessao_aberta"].sum()}')
print('  → per_capita menos confiável (sem lotação física definível)')

# Locais semifixos: capacidade VARIA por formato (válido, mas reportar)
_RE_SEMI = r'ginásio|ginasio|quadra|campo|arena|piscina|área aquática'
_sess['eh_semifixo'] = (
    _sess['localNome'].fillna('').str.lower().str.contains(_RE_SEMI)
    | _sess['TipologiaLocal'].fillna('').str.lower().str.contains(_RE_SEMI)
)
_ativ_semi = _sess.groupby('atividade_id')['eh_semifixo'].any().reset_index()
_ativ_semi.columns = ['atividade_id', 'tem_sessao_semifixo']
print(f'\n─── Atividades com ≥1 sessão em local semifixo (ginásio/quadra/piscina) ───')
print(f'  Total: {_ativ_semi["tem_sessao_semifixo"].sum()}')
print('  NOTA: capacidade variável por formato é ESPERADA. Reportamos para')
print('        saber quantas atividades dependem de um único valor de a.lugares.')

In [ ]:
# ── Diagnóstico D3: capacidade vs. tipo de local ────────────────────────
_diag = (
    raw_acoes_df[raw_acoes_df['atividade_id'].isin(_ids_validos)]
    [['atividade_id', 'a.lugares', 'a.estimativa_publico']]
    .merge(_ativ_abertos,  on='atividade_id', how='left')
    .merge(_ativ_semi,     on='atividade_id', how='left')
    .merge(_n_locais[['atividade_id', 'n_locais_distintos', 'n_tipos']],
           on='atividade_id', how='left')
)
_diag['lugares_n']    = pd.to_numeric(_diag['a.lugares'],            errors='coerce')
_diag['estimativa_n'] = pd.to_numeric(_diag['a.estimativa_publico'], errors='coerce')
_diag['ambos_nulos']  = _diag['lugares_n'].isna() & _diag['estimativa_n'].isna()
_diag['tem_sessao_aberta']   = _diag['tem_sessao_aberta'].fillna(False)
_diag['tem_sessao_semifixo'] = _diag['tem_sessao_semifixo'].fillna(False)

print('─── Capacidade nula × tipo de local ───')
_tab = _diag.groupby(['tem_sessao_aberta', 'tem_sessao_semifixo']).agg(
    total      =('atividade_id', 'count'),
    ambos_nulos=('ambos_nulos',  'sum'),
    pct_nulos  =('ambos_nulos',  'mean'),
).round(3).reset_index()
print(_tab.to_string(index=False))
print('  tem_sessao_aberta=True  → sem lotação física definível')
print('  tem_sessao_semifixo=True → capacidade variável por formato (normal)')

print('\n─── Variância de a.lugares em locais semifixos ───')
_semi_cap = _diag[
    _diag['tem_sessao_semifixo'] & _diag['lugares_n'].notna() & (_diag['lugares_n'] > 0)
]
print(_semi_cap['lugares_n'].describe(percentiles=[.25,.5,.75,.95]).round(0))
print('  NOTA: dispersão esperada — mesmo local, formatos diferentes (sentado/em pé).')

print('\n─── Top 20 TipologiaLocal para atividades com ambos nulos ───')
_ids_nulos = set(_diag[_diag['ambos_nulos']]['atividade_id'])
print(
    _sess[_sess['atividade_id'].isin(_ids_nulos)]
    ['TipologiaLocal'].value_counts().head(20).to_string()
)

In [ ]:
# ── Diagnóstico D4: serviços per-capita obrigatórios × capacidade ────────
_SERVICOS_PER_CAPITA = {'Apresentação', 'Curso', 'Palestra', 'Oficina'}

_pc = (
    raw_acoes_df[raw_acoes_df['atividade_id'].isin(_ids_validos)]
    [['atividade_id', 'a.lugares', 'a.estimativa_publico', 'servico']].copy()
)
_pc['lugares_n']    = pd.to_numeric(_pc['a.lugares'],            errors='coerce')
_pc['estimativa_n'] = pd.to_numeric(_pc['a.estimativa_publico'], errors='coerce')
_pc['ambos_nulos']  = _pc['lugares_n'].isna() & _pc['estimativa_n'].isna()
_pc['lugares_zero'] = _pc['lugares_n'].fillna(0) <= 0
_pc['estim_zero']   = _pc['estimativa_n'].fillna(0) <= 0

_pc = _pc.merge(_ativ_abertos, on='atividade_id', how='left')
_pc['tem_sessao_aberta'] = _pc['tem_sessao_aberta'].fillna(False)

# capacidade_confiavel=False: ambos nulos, ambos zero, ou local aberto
# locais semifixos NÃO são incluídos — capacidade variável é válida
_pc['capacidade_confiavel'] = ~(
    _pc['ambos_nulos']
    | (_pc['lugares_zero'] & _pc['estim_zero'])
    | _pc['tem_sessao_aberta']
)

_pc_obrig = _pc[_pc['servico'].isin(_SERVICOS_PER_CAPITA)].copy()

print('─── Serviços per-capita obrigatórios — capacidade_confiavel=False ───')
_resumo = (
    _pc_obrig.groupby('servico')
    .agg(
        total         =('atividade_id',        'count'),
        sem_capacidade=('capacidade_confiavel', lambda x: (~x).sum()),
        pct_problema  =('capacidade_confiavel', lambda x: (~x).mean()),
    )
    .round(3)
    .sort_values('total', ascending=False)
    .reset_index()
)
_resumo['pct_%'] = (_resumo['pct_problema'] * 100).round(1).astype(str) + '%'
print(_resumo.drop(columns='pct_problema').to_string(index=False))

total_obrig    = len(_pc_obrig)
total_problema = (~_pc_obrig['capacidade_confiavel']).sum()
print(f'\n─── TOTAL ───')
print(f'  Atividades per-capita obrigatórias:  {total_obrig}')
print(f'  Com capacidade_confiavel=False:       {total_problema}  ({total_problema/total_obrig:.1%})')

print('\n─── Detalhamento (colunas individuais) ───')
print(f'  Ambos nulos:               {_pc_obrig["ambos_nulos"].sum()}  ({_pc_obrig["ambos_nulos"].mean():.1%})')
print(f'  Tem sessão aberta/externa: {_pc_obrig["tem_sessao_aberta"].sum()}  ({_pc_obrig["tem_sessao_aberta"].mean():.1%})')
print(f'  Ambos zero:                {(_pc_obrig["lugares_zero"] & _pc_obrig["estim_zero"]).sum()}')

print('\n─── Top 10 TipologiaLocal para estes serviços com capacidade_confiavel=False ───')
_ids_problema = set(_pc_obrig[~_pc_obrig['capacidade_confiavel']]['atividade_id'])
print(
    _sess[_sess['atividade_id'].isin(_ids_problema)]
    ['TipologiaLocal'].value_counts().head(10).to_string()
)

In [15]:
print(raw_datas_sessoes_raw_df.columns[raw_datas_sessoes_raw_df.columns.duplicated()].tolist())

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 17, Finished, Available, Finished, False)

[]


In [16]:
# Todas as datas por atividade — formato "seg, dd/mm/yy HHhMM"
# Deduplica por (atividade_id, datainicio) antes de agregar
_d = (
    raw_datas_sessoes_df[['atividade_id', 'datainicio', 'diaSemanaTxt']]
    .drop_duplicates()
    .sort_values(['atividade_id', 'datainicio'])
    .assign(_linha=lambda d:
        d['diaSemanaTxt'] + ', ' + d['datainicio'].dt.strftime('%d/%m/%y %Hh%M')
    )
)

todas_as_datas_df = (
    _d.groupby('atividade_id')['_linha']
    .apply('\n'.join)
    .reset_index(name='todas_as_datas')
)

print(f'todas_as_datas_df: {todas_as_datas_df.shape}')
print()
print('Exemplo:')
print(todas_as_datas_df['todas_as_datas'].iloc[0])

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 18, Finished, Available, Finished, False)

todas_as_datas_df: (35272, 2)

Exemplo:
ter, 24/03/26 12h00
qua, 25/03/26 12h00
qui, 26/03/26 12h00


## 6. Projetos


In [17]:
raw_projetos_df = raw_projetos_df.drop_duplicates(subset=['projeto_id'])
for col in ['projeto_uo_nome']:
    if col in raw_projetos_df.columns:
        raw_projetos_df[col] = raw_projetos_df[col].fillna('').astype(str).str.strip()
print(f'raw_projetos_df: {raw_projetos_df.shape}')


StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 19, Finished, Available, Finished, False)

raw_projetos_df: (1299, 14)


## 7. Acessibilidade


In [18]:
if 'uo' not in raw_acessibilidade_df.columns:
    raw_acessibilidade_df = raw_acessibilidade_df.merge(
        raw_acoes_df[["atividade_id", "uo"]],
        on="atividade_id", how="left"
    )
raw_acessibilidade_df = raw_acessibilidade_df[
    raw_acessibilidade_df['atividade_id'].isin(raw_acoes_df['atividade_id'])
].copy()
print(f'raw_acessibilidade_df: {raw_acessibilidade_df.shape}')


StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 20, Finished, Available, Finished, False)

raw_acessibilidade_df: (2328, 5)


## 8. Solicitações


In [19]:
# ── tem_passagem: derivado de raw_solicitacoes_df (elimina sql_passagens) ─
# Equivalente a WHERE item_grupo LIKE '%Passagem%'; filtra custo > 0
# (passagens com custo = 0 não devem gerar autonomia STS)
_passagens_ids = raw_solicitacoes_df[
    raw_solicitacoes_df['item_grupo'].str.contains('Passagem', case=False, na=False)
]['atividade_id']

rps_parcial_df['tem_passagem'] = np.where(
    rps_parcial_df['atividade_id'].isin(_passagens_ids), 'Sim', 'Não'
)
print(f'tem_passagem=Sim: {(rps_parcial_df["tem_passagem"] == "Sim").sum()}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 21, Finished, Available, Finished, False)

tem_passagem=Sim: 588


## 9. Contratos


In [20]:
SERVICOS_LIMIAR_20K  = {'Debate', 'Seminário', 'Visita Mediada'}
SUBATIV_LIMIAR_20K   = {'Ações formativas', 'Ações mediadas', 'Passeios', 'Viagens'}
SERVICOS_POR_HORA    = {'Curso', 'Oficina', 'Vivência', 'Seminário', 'Mediação', 'Visita Mediada', 'Intervenção urbana'}
SUBATIV_POR_HORA     = {'Multipráticas recreativas', 'Passeios', 'Viagens', 'Colônias recreativas'}


def build_contracts(
    raw_solicitacoes_df: pd.DataFrame,
    rps_parcial_df: pd.DataFrame,
) -> pd.DataFrame:
    """Constrói contracts_df a partir de raw_solicitacoes_df.

    Substitui sql_contratos (5 subqueries Hive) por groupby Python sobre
    raw_solicitacoes_df, que já está disponível na seção 8.
    Saída: uma linha por solicitação; métricas agregadas repetidas por atividade_id.
    """
    solic = raw_solicitacoes_df.copy()

    # --- classificação de tipo por item_grupo / nome_item ---
    grupo_lower = solic['item_grupo'].fillna('').str.lower()
    item_lower  = solic['nome_item'].fillna('').str.lower()

    is_admin      = solic['area'].str.strip() == 'Administrativo'
    is_contrato   = grupo_lower.str.contains('contrato') | item_lower.str.contains('contrato')
    is_filme      = grupo_lower.str.contains('filme')
    is_passagem   = grupo_lower.str.contains('passagem') | item_lower.str.contains('passagem')
    is_hospedagem = grupo_lower.str.contains('hospedagem') | item_lower.str.contains('hospedagem')

    # Filtro principal: admin + (contrato | passagem | hospedagem | filme)
    # Espelha o WHERE do FROM principal da query Hive original
    mask_main = is_admin & (is_contrato | is_filme | is_passagem | is_hospedagem)
    df = solic[mask_main].copy()

    # Renomeia para manter compatibilidade com o código consumidor (build_autonomias, base)
    df = df.rename(columns={'item_grupo': 'grupo', 'nome_item': 'item', 'descricao': 'a.complemento'})
    # Remove sufixo "[...]" que a view do Hive às vezes insere no grupo
    df['grupo'] = df['grupo'].astype(str).str.split('[').str[0].str.strip()

    # --- agregados por atividade_id (substituem os 4 subqueries do Hive) ---

    # custo_contratos_total / n_contratos: apenas contrato+filme, área Administrativo
    mask_cf = is_admin & (is_contrato | is_filme)
    agg_cf = (
        solic[mask_cf]
        .groupby('atividade_id', as_index=False)
        .agg(custo_contratos_total=('custo', 'sum'), n_contratos=('solicitacao_id', 'count'))
    )

    # custo_total / n_solic: todas as solicitações (raw_solicitacoes_df já filtrou custo > 0)
    agg_total = (
        solic
        .groupby('atividade_id', as_index=False)
        .agg(custo_total=('custo', 'sum'), n_solic=('solicitacao_id', 'count'))
    )

    df = df.merge(agg_cf,    on='atividade_id', how='left')
    df = df.merge(agg_total, on='atividade_id', how='left')
    df[['custo_contratos_total', 'custo_total']] = (
        df[['custo_contratos_total', 'custo_total']].fillna(0)
    )
    df['n_contratos'] = df['n_contratos'].fillna(0).astype(int)
    df['n_solic']     = df['n_solic'].fillna(0).astype(int)


    # --- flags de custo INDIVIDUAL (custo desta solicitação, não do total da atividade) ---
    custo_ind = df['custo']
    bool_map  = {1: 'sim', 0: '0'}
    df['acima15mil']  = (custo_ind > 15000).astype(int).map(bool_map)
    df['acima20mil']  = (custo_ind > 20000).astype(int).map(bool_map)
    df['acima100mil'] = (custo_ind > 100000).astype(int).map(bool_map)


    # --- servico e subatividade para classificar autonomiaCusto ---
    df = df.merge(
        rps_parcial_df[['atividade_id', 'servico', 'subatividade']],
        on='atividade_id', how='left',
    )
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    # --- autonomiaCusto baseada no custo INDIVIDUAL desta solicitação ---
    # build_autonomias resolve a hierarquia final tomando drop_duplicates; aqui classificamos por linha
    limiar_20k = (
        df['servico'].isin(SERVICOS_LIMIAR_20K) |
        df['subatividade'].isin(SUBATIV_LIMIAR_20K)
    )
    acima15  = df['acima15mil']  == 'sim'
    acima20  = df['acima20mil']  == 'sim'
    acima100 = df['acima100mil'] == 'sim'

    df['autonomiaCusto'] = np.select(
        [acima100,
         acima20 &  limiar_20k,
         acima20 & ~limiar_20k,
         acima15 & ~limiar_20k],
        ['DIREG', 'STS', 'STS-20', 'STS-15'],
        default='UO',
    )


    return df.drop_duplicates(subset=['solicitacao_id'])

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 22, Finished, Available, Finished, False)

In [21]:
# contracts_df: tabela de solicitações administrativas com métricas de custo.
# Executado aqui porque build_contracts() depende de:
#   - raw_solicitacoes_df  (disponível após a seção 8)
#   - rps_parcial_df com tem_passagem já adicionado (rps-passagens-exec acima)
contracts_df = build_contracts(
    raw_solicitacoes_df = raw_solicitacoes_df,
    rps_parcial_df      = rps_parcial_df,
)
print(f'contracts_df:     {contracts_df.shape}')
print(f'atividades unicas: {contracts_df["atividade_id"].nunique()}')
print()
print('autonomiaCusto:')
print(contracts_df.drop_duplicates("atividade_id")["autonomiaCusto"].value_counts())

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 23, Finished, Available, Finished, False)

contracts_df:     (20266, 30)
atividades unicas: 15469

autonomiaCusto:
autonomiaCusto
UO        12643
STS-20     1637
STS-15      536
STS         420
DIREG       233
Name: count, dtype: int64


## Solicitações


In [22]:
_ITEM_PARA_GRUPO = {
    'Ação esportiva e recreativa - PJ':              'Contrato PJ',
    'Água - Sessão':                                  'Água',
    'Aquisição de material - Cultural Educativo':     'Compras',
    'Aquisição de material - Curso/Palestra etc':     'Compras',
    'Aquisição de material - Esportivo e Recreativo': 'Compras',
    'Audiovisual - Projeção Cine - Sessão':           'Audiovisual / Projeção',
    'Audiovisual - Sonorização - Sessão':             'Sonorização',
    'Camarim Tipo 1 - Sessão':                        'Camarim',
    'Camarim Tipo 2 - Sessão':                        'Camarim',
    'Coffee break Tipo 1 - Sessão':                   'Café para integrações/Ações Similares',
    'Coffee break Tipo 2 - Sessão':                   'Café para integrações/Ações Similares',
    'Contrato PF - Criação artística':                'Contrato PF',
    'Contrato PF - Curso/Palestra etc':               'Contrato PF',
    'Contrato PJ - Ação artística':                   'Contrato PJ',
    'Contrato PJ - Apresentação Esportiva':           'Contrato PJ',
    'Contrato PJ - Curso/Palestra etc':               'Contrato PJ',
    'Guia de turismo':                                'Turismo',
    'Hospedagem':                                     'Hospedagem',
    'Kit Lanche - Sessão':                            'Kit Lanche',
    'Locação - Equipamento de sonorização':           'Locação - Sonorização',
    'Locação - Eventual - Diversos':                  'Locação - Outros',
    'Locação de Automóvel sem Motorista':             'Transporte',
    'Recepções / Refeições':                          'Diversos - Alimentação',
    'Serviço de arbitragem':                          'Contrato PJ',
    'Serviço de bombeiro civil':                      'Serviços Gerais Terceirizados',
    'Serviço de Montagem de Mobiliário/Equipamento':  'Montagem',
    'Transporte - Para Turismo Social':               'Turismo',
    'Transporte - Pessoas - exceto turismo':          'Transporte',
    'Transporte - Sessão':                            'Transporte',
    'Verificar':                                      'Outros',
}

_GRUPO_PARA_MACRO = {
    'Água':                                   'Alimentação',
    'Brindes':                                'Alimentação',
    'Café para integrações/Ações Similares':  'Alimentação',
    'Camarim':                                'Alimentação',
    'Diversos - Alimentação':                 'Alimentação',
    'Kit Lanche':                             'Alimentação',
    'Serviço de Receptivo':                   'Alimentação',
    'Diversos - Comunicação':                 'Comunicação e Editorial',
    'Editoria web':                           'Comunicação e Editorial',
    'Impressos e digitais':                   'Comunicação e Editorial',
    'Contrato PF':                            'Contrato PF',
    'Contratações diversas':                  'Contrato PJ',
    'Contrato Cooperativa':                   'Contrato PJ',
    'Contrato PJ':                            'Contrato PJ',
    'Exibição de Filmes':                     'Contrato PJ',
    'Hospedagem':                             'Hospedagem',
    'Compras':                                'Outros',
    'Elétrica':                               'Outros',
    'Equipamentos':                           'Outros',
    'Locação - Outros':                       'Outros',
    'Mobiliário':                             'Outros',
    'Montagem':                               'Outros',
    'Outros':                                 'Outros',
    'Outros - Internos':                      'Outros',
    'Outros - Terceiros':                     'Outros',
    'Passagem Aérea':                         'Passagem',
    'Acompanhamento':                         'Serviços Operacionais',
    'Limpeza':                                'Serviços Operacionais',
    'Serviços Gerais Terceirizados':          'Serviços Operacionais',
    'Acessibilidade':                         'Serviços Técnicos',
    'Audiovisual / Projeção':                 'Serviços Técnicos',
    'Iluminação':                             'Serviços Técnicos',
    'Locação - Iluminação':                   'Serviços Técnicos',
    'Locação - Sonorização':                  'Serviços Técnicos',
    'Sonorização':                            'Serviços Técnicos',
    'Tradução Simultânea':                    'Serviços Técnicos',
    'Transporte':                             'Transporte',
    'Turismo':                                'Turismo',
}


def build_solicitacoes(
    raw_solicitacoes_df: pd.DataFrame,
    rps_parcial_df: pd.DataFrame,
) -> pd.DataFrame:
    """Constrói solicitacoes_df: todas as solicitações (sem filtro de custo).

    Diferenças em relação a contracts_df:
    - Inclui todas as solicitações, independente de custo
    - sem servico/subatividade no output (estão em tabela_base)
    - custo_solic_filtradas: soma do subconjunto filtrado por atividade_id
    Métricas de custo por sessão/per capita implementadas como medidas DAX no modelo semântico.
    """
    solic = raw_solicitacoes_df.copy()

    # ── classificação de tipo ──────────────────────────────────────────────
    grupo_lower = solic['item_grupo'].fillna('').str.lower()
    item_lower  = solic['nome_item'].fillna('').str.lower()

    is_admin      = solic['area'].str.strip() == 'Administrativo'
    is_contrato   = grupo_lower.str.contains('contrato') | item_lower.str.contains('contrato')
    is_filme      = grupo_lower.str.contains('filme')
    is_passagem   = grupo_lower.str.contains('passagem') | item_lower.str.contains('passagem')
    is_hospedagem = grupo_lower.str.contains('hospedagem') | item_lower.str.contains('hospedagem')

    df = solic.copy()

    df = df.rename(columns={'item_grupo': 'grupo', 'nome_item': 'item', 'descricao': 'a.complemento'})
    df['grupo'] = df['grupo'].astype(str).str.split('[').str[0].str.strip()
    df['grupo'] = df['grupo'].replace({'null': pd.NA, 'None': pd.NA, '': pd.NA})
    _item_key = df['item'].astype(str).str.split('[').str[0].str.strip()
    _mask = df['grupo'].isna()
    df.loc[_mask, 'grupo'] = _item_key[_mask].map(_ITEM_PARA_GRUPO)

    # ── agregados por atividade_id ─────────────────────────────────────────

    # custo_contratos_total / n_contratos: contrato+filme admin (semântica original)
    mask_cf = is_admin & (is_contrato | is_filme)
    agg_cf = (
        solic[mask_cf]
        .groupby('atividade_id', as_index=False)
        .agg(custo_contratos_total=('custo', 'sum'), n_contratos=('solicitacao_id', 'count'))
    )

    # custo_total / n_solic: todas as solicitações
    agg_total = (
        solic
        .groupby('atividade_id', as_index=False)
        .agg(custo_total=('custo', 'sum'), n_solic=('solicitacao_id', 'count'))
    )

    # custo_solic_filtradas: soma das que passaram no filtro (por atividade)
    agg_filtradas = (
        df.groupby('atividade_id', as_index=False)
        .agg(custo_solic_filtradas=('custo', 'sum'))
    )

    df = df.merge(agg_cf,        on='atividade_id', how='left')
    df = df.merge(agg_total,     on='atividade_id', how='left')
    df = df.merge(agg_filtradas, on='atividade_id', how='left')
    df[['custo_contratos_total', 'custo_total', 'custo_solic_filtradas']] = (
        df[['custo_contratos_total', 'custo_total', 'custo_solic_filtradas']].fillna(0)
    )
    df['n_contratos'] = df['n_contratos'].fillna(0).astype(int)
    df['n_solic']     = df['n_solic'].fillna(0).astype(int)


    # ── flags de custo individual ──────────────────────────────────────────
    bool_map = {1: 'sim', 0: '0'}
    df['acima15mil']  = (df['custo'] > 15000 ).astype(int).map(bool_map)
    df['acima20mil']  = (df['custo'] > 20000 ).astype(int).map(bool_map)
    df['acima100mil'] = (df['custo'] > 100000).astype(int).map(bool_map)


    # ── servico/subatividade: uso INTERNO para autonomiaCusto ──
    df = df.merge(
        rps_parcial_df[['atividade_id', 'servico', 'subatividade']],
        on='atividade_id', how='left',
    )
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    # ── autonomiaCusto (por solicitação individual) ────────────────────────
    limiar_20k = (
        df['servico'].isin(SERVICOS_LIMIAR_20K) |
        df['subatividade'].isin(SUBATIV_LIMIAR_20K)
    )
    acima15  = df['acima15mil']  == 'sim'
    acima20  = df['acima20mil']  == 'sim'
    acima100 = df['acima100mil'] == 'sim'
    df['autonomiaCusto'] = np.select(
        [acima100,
         acima20 &  limiar_20k,
         acima20 & ~limiar_20k,
         acima15 & ~limiar_20k],
        ['DIREG', 'STS', 'STS-20', 'STS-15'],
        default='UO',
    )


    # ── remove campos internos (estão em tabela_base) ─────────────────────
    df = df.drop(columns=['servico', 'subatividade'])


    # -- alerta: complemento mal preenchido ------------------------------------
    _grp  = df['grupo'].fillna('').str.lower()
    _itm  = df['item'].fillna('').str.lower()
    _comp = df['a.complemento'].fillna('')

    _is_pass = _grp.str.contains('passagem')   | _itm.str.contains('passagem')
    _is_hosp = _grp.str.contains('hospedagem') | _itm.str.contains('hospedagem')
    _is_cont = _grp.str.contains('contrato')   | _itm.str.contains('contrato')

    _pass_ok = _comp.map(_valid_passagem)
    _hosp_ok = _comp.map(_valid_hospedagem)
    _pcap_ok = _comp.str.contains(_RE_PCAP_CHECK.pattern,     flags=re.IGNORECASE, na=False)

    df['alerta'] = (
        (_is_pass & ~_pass_ok) |
        (_is_hosp & ~_hosp_ok) |
        (_is_cont & ~_pcap_ok)
    ).astype(int)

    df['macro_grupo'] = df['grupo'].map(_GRUPO_PARA_MACRO)
    return df.drop_duplicates(subset=['solicitacao_id'])


StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 24, Finished, Available, Finished, False)

In [23]:
# solicitacoes_df: todas as solicitações (sem filtro de custo).
solicitacoes_df = build_solicitacoes(
    raw_solicitacoes_df = raw_solicitacoes_df,
    rps_parcial_df      = rps_parcial_df,
)
print(f'solicitacoes_df:   {solicitacoes_df.shape}')
print(f'atividades unicas: {solicitacoes_df["atividade_id"].nunique()}')


StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 25, Finished, Available, Finished, False)

solicitacoes_df:   (23296, 32)
atividades unicas: 16595


## 10. Autonomias


In [24]:
# Hierarquia: DIREG (3) > STS (2) > UO (1)
# Cada fonte de autonomia e convertida para nivel numerico;
# o nivel maximo determina a autonomia final.


NIVEL = {'DIREG': 3, 'STS': 2, 'STS-20': 2, 'STS-15': 2, 'UO': 1}


def autonomia_nivel(serie: pd.Series) -> pd.Series:
    """Converte valores de autonomia para nivel numerico (desconhecido 1)."""
    return serie.map(NIVEL).fillna(1).astype(int)


NIVEL_INV = {3: 'DIREG', 2: 'STS', 1: 'UO'}


def build_autonomias(rps_df, datas_df, contracts_df, raw_pcap_df) -> pd.DataFrame:

    # Base: uma linha por atividade com servico cadastrado
    df = rps_df[['atividade_id', 'servico', 'subatividade', 'tag', 'tem_passagem']].copy()

    # --- autonomia temporal (ja ajustada por servico em Datas) ---
    df = df.merge(
        datas_df[['atividade_id', 'autonomiaTemporal', '60horas', '90dias', '30dias', 'diascorridos']],
        on='atividade_id', how='left'
    )
    df['autonomiaTemporal'] = df['autonomiaTemporal'].fillna('UO')
    df['60horas']           = df['60horas'].fillna('0')
    df['90dias']            = df['90dias'].fillna('0')
    df['30dias']            = df['30dias'].fillna('0')
    df['diascorridos']      = pd.to_numeric(df['diascorridos'], errors='coerce').fillna(0)

    # --- autonomia de custo (uma linha por atividade, prevalece o maior nivel) ---
    custo_por_ativ = (
        contracts_df[['atividade_id', 'autonomiaCusto']]
        .assign(_nivel=lambda d: d['autonomiaCusto'].map(NIVEL).fillna(1))
        .sort_values('_nivel', ascending=False)
        .drop_duplicates(subset=['atividade_id'])
        .drop(columns=['_nivel'])
    )
    df = df.merge(custo_por_ativ, on='atividade_id', how='left')
    df['autonomiaCusto'] = df['autonomiaCusto'].fillna('UO')

    # --- autonomia por PCAP ---
    # Para cada atividade, aplica os mesmos limiares de autonomiaCusto ao pcap_total de cada PCAP vinculada.
    # Se ao menos uma PCAP atingir um limiar, o nivel mais alto entre todas as PCAPs e considerado.
    _pcap_aut = pd.DataFrame(columns=['atividade_id', 'autonomiaPCAP'])
    if raw_pcap_df is not None and len(raw_pcap_df) > 0 and 'pcap_total' in raw_pcap_df.columns:
        _pcap_vals = raw_pcap_df[['atividade_id', 'pcap_total']].copy()
        _pcap_vals['pcap_total'] = pd.to_numeric(_pcap_vals['pcap_total'], errors='coerce').fillna(0)
        _pcap_ext = (
            df[['atividade_id', 'servico', 'subatividade']]
            .merge(_pcap_vals, on='atividade_id', how='inner')
        )
        _limiar_20k_p = (
            _pcap_ext['servico'].isin(SERVICOS_LIMIAR_20K) |
            _pcap_ext['subatividade'].isin(SUBATIV_LIMIAR_20K)
        )
        _pcap_ext['autonomiaPCAP'] = np.select(
            [_pcap_ext['pcap_total'] > 100_000,
             (_pcap_ext['pcap_total'] > 20_000) &  _limiar_20k_p,
             (_pcap_ext['pcap_total'] > 20_000) & ~_limiar_20k_p,
             (_pcap_ext['pcap_total'] > 15_000) & ~_limiar_20k_p],
            ['DIREG', 'STS', 'STS-20', 'STS-15'],
            default='UO',
        )
        _pcap_aut = (
            _pcap_ext[['atividade_id', 'autonomiaPCAP']]
            .assign(_nivel=lambda d: d['autonomiaPCAP'].map(NIVEL).fillna(1))
            .sort_values('_nivel', ascending=False)
            .drop_duplicates(subset=['atividade_id'])
            .drop(columns=['_nivel'])
        )
    df = df.merge(_pcap_aut, on='atividade_id', how='left')
    df['autonomiaPCAP'] = df['autonomiaPCAP'].fillna('UO')

    # --- regra de custo para 60h (baseada em contratos; PCAP nao altera o rebaixamento) ---
    custo_acima20 = df['autonomiaCusto'].isin(['STS', 'STS-20', 'DIREG'])

    mask_rebaixa = ~custo_acima20 & (df['autonomiaTemporal'] == 'DIREG') & (df['90dias'] != 'sim')
    _sts = (
        (df.loc[mask_rebaixa, '30dias'] == 'sim') |
        (df.loc[mask_rebaixa, 'diascorridos'] > 30)
    )
    df.loc[mask_rebaixa, 'autonomiaTemporal'] = np.where(_sts, 'STS', 'UO')

    # --- niveis por fonte ---
    nivel_temporal  = autonomia_nivel(df['autonomiaTemporal'])
    nivel_custo     = autonomia_nivel(df['autonomiaCusto'])
    nivel_pcap      = autonomia_nivel(df['autonomiaPCAP'])
    nivel_tag       = np.where(df['tag'] == 'Avaliacao STS', 2, 1)
    nivel_passagem  = np.where(df['tem_passagem'] == 'Sim',  2, 1)

    # --- regra combinada: 60h + custo >= 20k -> DIREG ---
    nivel_combinado = np.where(
        (df['60horas'] == 'sim') & custo_acima20,
        3, 1
    )

    # --- autonomia final: prevalece o maior nivel (DIREG > STS > UO) ---
    nivel_final = pd.concat(
        [nivel_temporal, nivel_custo, nivel_pcap,
         pd.Series(nivel_tag,       index=df.index),
         pd.Series(nivel_passagem,  index=df.index),
         pd.Series(nivel_combinado, index=df.index)],
        axis=1
    ).max(axis=1)

    df['autonomia'] = nivel_final.map(NIVEL_INV)

    # --- diagnostico: atividades cuja autonomia foi elevada pelo valor da PCAP ---
    _nivel_sem_pcap = pd.concat(
        [nivel_temporal, nivel_custo,
         pd.Series(nivel_tag,       index=df.index),
         pd.Series(nivel_passagem,  index=df.index),
         pd.Series(nivel_combinado, index=df.index)],
        axis=1
    ).max(axis=1)
    _elev = nivel_pcap > _nivel_sem_pcap
    _n_direg_pcap = int((_elev & (nivel_final == 3)).sum())
    _n_sts_pcap   = int((_elev & (nivel_final == 2)).sum())
    print(f'Via PCAP -> DIREG: {_n_direg_pcap} atividade(s)')
    print(f'Via PCAP -> STS:   {_n_sts_pcap} atividade(s)')

    return df.drop(columns=['60horas', '90dias', '30dias', 'diascorridos'])

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 26, Finished, Available, Finished, False)

## 11. Descrição de Custos


In [25]:
# Itens a descartar antes de construir item_desc
ITENS_DESCARTAR = frozenset([
    'Camarim', 'Camarim Tipo 1', 'Camarim Tipo 2',
    'Água', 'Verificar',
])

# Mapeamento exato: item_de_custo → categoria normalizada
MAPA_ITEM_CUSTO = {
    # Contratos
    'Contrato PJ [Custo cachê/pró-labore]':            'Contrato PJ',
    'Contrato PJ':                                      'Contrato PJ',
    'Contrato PF [Custo cachê/pró-labore]':            'Contrato PF',
    'Contrato PF':                                      'Contrato PF',
    'Contrato Cooperativa [Custo cachê/pró-labore]':   'Contrato Cooperativa',
    # Viagem
    'Hospedagem [Custo hospedagem]':                   'Hospedagem',
    'Passagem Aérea [Custo passagem]':                 'Passagem Aérea',
    'Turismo [Outros custos de terceiros]':            'Turismo',
    'Turismo':                                          'Turismo',
    'Transporte [Outros custos de terceiros]':         'Transporte',
    'Transporte':                                       'Transporte',
    # Serviços artísticos/técnicos
    'Exibição de Filmes':                              'Exibição de Filmes',
    'Ação esportiva e recreativa':                     'Ação esportiva e recreativa',
    'Contratações diversas [Outros custos de terceiros]': 'Contratações diversas',
    # Compras
    'Compras  [Outros custos internos]':               'Compras',
    'Compras':                                          'Compras',
    'Aquisição de material':                           'Compras',
    # Alimentação
    'Kit Lanche':                                       'Alimentação',
    'Brindes':                                          'Alimentação',
    'Café para integrações/Ações Similares':           'Alimentação',
    'Coffee break Tipo 1':                             'Alimentação',
    'Coffee break Tipo 2':                             'Alimentação',
    'Coquetel':                                         'Alimentação',
    'Recepções / Refeições':                           'Alimentação',
    'Diversos - Alimentação [Outros custos de terceiros]': 'Alimentação',
    # Infraestrutura de evento
    'Sonorização':                                      'Sonorização',
    'Locação - Sonorização [Outros custos de terceiros]': 'Sonorização',
    'Iluminação':                                       'Iluminação',
    'Locação - Iluminação [Outros custos de terceiros]': 'Iluminação',
    'Audiovisual / Projeção':                          'Audiovisual',
    'Audiovisual':                                      'Audiovisual',
    'Locação - Outros [Outros custos de terceiros]':   'Locação',
    'Locação':                                          'Locação',
    # Comunicação
    'Impressos e digitais':                            'Comunicação',
    'Editoria web':                                     'Comunicação',
    'Assessoria de imprensa':                          'Comunicação',
    'Diversos - Comunicação [Outros custos de terceiros]': 'Comunicação',
    # Acessibilidade
    'Tradução Simultânea':                             'Acessibilidade',
    # Outros (categorias absorvidas)
    'Serviço de Receptivo':                            'Outros',
    'Mobiliário':                                       'Outros',
    'Montagem':                                         'Outros',
    'Equipamentos':                                     'Outros',
    'Limpeza':                                          'Outros',
    'Elétrica':                                         'Outros',
    'Acompanhamento':                                   'Outros',
    'Outros':                                           'Outros',
    'Outros - Terceiros [Outros custos terceiros]':    'Outros',
    'Outros - Internos [Outros custos internos]':      'Outros',
}

# Fallback por prefixo — para itens com código de conta no nome (ex: 'Hospedagem [3920')
PREFIXOS_ITEM_CUSTO = [
    ('Hospedagem',           'Hospedagem'),
    ('Guia de turismo',      'Turismo'),
    ('Locação de Automóvel', 'Transporte'),
    ('Serviço de Montagem',  'Outros'),
    ('Serviços Gerais',      'Outros'),
]


def normalizar_item_custo(idc: str, item_grupo: str) -> str:
    """Retorna categoria normalizada; None = descartar."""
    idc_s   = str(idc).strip()
    grupo_l = str(item_grupo).strip().lower()
    # Estagiário → descarte
    if 'estagi' in grupo_l or 'estagi' in idc_s.lower():
        return None
    # item_grupo = 'acessibilidade' → Acessibilidade
    if grupo_l == 'acessibilidade':
        return 'Acessibilidade'
    # Match exato
    if idc_s in MAPA_ITEM_CUSTO:
        return MAPA_ITEM_CUSTO[idc_s]
    # Fallback por prefixo
    for prefixo, categoria in PREFIXOS_ITEM_CUSTO:
        if idc_s.startswith(prefixo):
            return categoria
    # Sem mapeamento — mantém o valor bruto
    return idc_s


StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 27, Finished, Available, Finished, False)

In [26]:
# Ordem de exibição dos grupos em item_desc
# 0 = Contratos  1 = Passagem Aérea  2 = Hospedagem  3 = demais (alfabético)
GRUPO_ITEM_DESC = {
    'Contrato PJ':          0,
    'Contrato PF':          0,
    'Contrato Cooperativa': 0,
    'Passagem Aérea':       1,
    'Hospedagem':           2,
}
SEP_ITEM_DESC = chr(9472) * 28   # ────────────────────────────


def build_solicitacoes_desc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── item_de_custo bruto ───────────────────────────────────────────────
    grupo = df['item_grupo'].fillna('').astype(str).str.strip()
    nome  = df['nome_item'].fillna('').astype(str).str.strip()
    nome_ate_traco = nome.str.split('-').str[0].str.strip()
    usar_nome = (grupo == '') | (grupo.str.lower() == 'null')
    df['item_de_custo'] = np.where(usar_nome, nome_ate_traco, grupo)

    # ── descarte ─────────────────────────────────────────────────────────
    df = df[~df['item_de_custo'].isin(ITENS_DESCARTAR)].copy()

    # ── normalização ─────────────────────────────────────────────────────
    df['item_norm'] = df.apply(
        lambda r: normalizar_item_custo(r['item_de_custo'], r['item_grupo']),
        axis=1,
    )
    df = df[df['item_norm'].notna()].copy()

    # ── chave de ordenação: (grupo, item_norm alfabético, solicitacao_id) ─
    df['_grp'] = df['item_norm'].map(lambda x: GRUPO_ITEM_DESC.get(x, 3))

    # ── custo formatado ───────────────────────────────────────────────────
    df['custo_fmt'] = df['custo'].apply(
        lambda x: 'R$ ' + f'{x:,.0f}'.replace(',', '.')
    )

    # ── linha individual ──────────────────────────────────────────────────
    desc = df['descricao'].fillna('').astype(str).str.strip()
    df['linha'] = df['item_norm'] + ' — ' + desc + ' — ' + df['custo_fmt']

    # ── agrega por atividade_id com separadores entre grupos ─────────────
    def montar_desc(sub):
        sub = sub.sort_values(['_grp', 'item_norm', 'solicitacao_id'])
        linhas = []
        grp_atual = None
        for _, row in sub.iterrows():
            if grp_atual is not None and row['_grp'] != grp_atual:
                linhas.append(SEP_ITEM_DESC)
            linhas.append(row['linha'])
            grp_atual = row['_grp']
        return chr(10).join(linhas)

    def agrupar(sub_df, col_name: str) -> pd.DataFrame:
        if sub_df.empty:
            return pd.DataFrame(columns=['atividade_id', col_name])
        return (
            sub_df.groupby('atividade_id')
            .apply(montar_desc)
            .reset_index(name=col_name)
        )

    # ── item_desc: exclui camarim, água e custo < R$100
    #   exceto contratos/passagens/hospedagens (_grp <= 2), que entram sempre
    _nome_lower = df['nome_item'].fillna('').str.lower()
    _item_lower = df['item_de_custo'].fillna('').str.lower()
    _is_camarim_agua = (
        _nome_lower.str.contains(r'camarim|água|agua', regex=True) |
        _item_lower.str.contains(r'camarim|água|agua', regex=True)
    )
    _is_relevante = (df['_grp'] <= 2) | (df['custo'] >= 100)
    df_item_desc = df[~_is_camarim_agua & _is_relevante]

    # base: todos os atividade_id com pelo menos uma solicitação
    resultado = df['atividade_id'].drop_duplicates().to_frame()

    if not df_item_desc.empty:
        resultado = resultado.merge(
            df_item_desc.groupby('atividade_id').apply(montar_desc).reset_index(name='item_desc'),
            on='atividade_id', how='left',
        )
    else:
        resultado['item_desc'] = None

    # ── descrições por categoria (usa df completo) ────────────────────────
    resultado = (
        resultado
        .merge(agrupar(df[df['_grp'] == 0],                    'contratos_desc'), on='atividade_id', how='left')
        .merge(agrupar(df[df['item_norm'] == 'Passagem Aérea'], 'passagem_desc'),  on='atividade_id', how='left')
        .merge(agrupar(df[df['item_norm'] == 'Hospedagem'],     'hospedagem_desc'), on='atividade_id', how='left')
    )

    # ── flags 0/1 ─────────────────────────────────────────────────────────
    resultado['tem_contrato']   = resultado['contratos_desc'].notna().astype(int)
    resultado['tem_passagem']   = resultado['passagem_desc'].notna().astype(int)
    resultado['tem_hospedagem'] = resultado['hospedagem_desc'].notna().astype(int)
    resultado['alerta'] = resultado['atividade_id'].map(
        df.groupby('atividade_id')['alerta'].max()
    ).fillna(0).astype(int)

    for col in ['contratos_desc', 'passagem_desc', 'hospedagem_desc']:
        resultado[col] = resultado[col].fillna('')

    return resultado


solicitacoes_desc_df = build_solicitacoes_desc(raw_solicitacoes_df)

print(f'solicitacoes_desc_df: {solicitacoes_desc_df.shape}')
print(f'  tem_contrato=1:   {solicitacoes_desc_df["tem_contrato"].sum()}')
print(f'  tem_passagem=1:   {solicitacoes_desc_df["tem_passagem"].sum()}')
print(f'  tem_hospedagem=1: {solicitacoes_desc_df["tem_hospedagem"].sum()}')
print()
print('Exemplo (primeira linha de item_desc):')
print(solicitacoes_desc_df['item_desc'].iloc[0])

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 28, Finished, Available, Finished, False)

solicitacoes_desc_df: (17488, 8)
  tem_contrato=1:   15011
  tem_passagem=1:   588
  tem_hospedagem=1: 2982

Exemplo (primeira linha de item_desc):
Contrato PJ — Luiza Helena Craemer Franscesconi - ME — R$ 8.775


## 12. PCAP


In [27]:
pcap_re       = r'PCAP[^0-9]*([0-9]{13})'    # com captura — para str.extract
pcap_re_check = r'PCAP[^0-9]*(?:[0-9]{13})'  # sem captura — para str.contains

# ── 1. busca em descricao (complemento) da solicitacao ───────────────────
sol_pcap = raw_solicitacoes_df[
    (raw_solicitacoes_df['area'] == 'Administrativo') &
    raw_solicitacoes_df['descricao'].str.contains(pcap_re_check, flags=re.IGNORECASE, na=False)
][['atividade_id', 'solicitacao_id', 'descricao']].copy()

sol_pcap['pcap_num'] = (
    sol_pcap['descricao']
    .str.extract(pcap_re, flags=re.IGNORECASE)[0]
    .pipe(pd.to_numeric, errors='coerce')
    .astype('Int64')
)
sol_pcap = sol_pcap.drop(columns=['descricao']).drop_duplicates(subset=['solicitacao_id'])

# ── 2. busca nos campos da justificativa (nivel atividade_id) ─────────────
_txt_cols_pcap = [c for c in ['sinopse_curta', 'sinopse_aprovacao', 'info_parceria', 'just_recursos']
                  if c in raw_acoes_txts_df.columns]

_txts = (
    raw_acoes_txts_df[['acao.atividade_id'] + _txt_cols_pcap]
    .rename(columns={'acao.atividade_id': 'atividade_id'})
    .drop_duplicates(subset=['atividade_id'])
    .copy()
)
_txts['_texto'] = _txts[_txt_cols_pcap].fillna('').apply(' '.join, axis=1)

just_pcap = _txts[
    _txts['_texto'].str.contains(pcap_re_check, flags=re.IGNORECASE, na=False)
][['atividade_id', '_texto']].copy()

just_pcap['pcap_num'] = (
    just_pcap['_texto']
    .str.extract(pcap_re, flags=re.IGNORECASE)[0]
    .pipe(pd.to_numeric, errors='coerce')
    .astype('Int64')
)
just_pcap = just_pcap.drop(columns=['_texto']).dropna(subset=['pcap_num'])

# Exclui atividades ja encontradas via descricao
just_pcap = just_pcap[~just_pcap['atividade_id'].isin(set(sol_pcap['atividade_id']))]

# Associa a melhor solicitacao_id disponivel: prefere area Administrativo
_solic_best = (
    pd.concat([
        raw_solicitacoes_df[raw_solicitacoes_df['area'] == 'Administrativo'][['atividade_id', 'solicitacao_id']].assign(_prio=0),
        raw_solicitacoes_df[['atividade_id', 'solicitacao_id']].assign(_prio=1),
    ])
    .sort_values(['atividade_id', '_prio'])
    .drop_duplicates(subset=['atividade_id'])
    [['atividade_id', 'solicitacao_id']]
)
just_pcap = just_pcap.merge(_solic_best, on='atividade_id', how='left')

print(f'PCAP via complemento:   {sol_pcap["atividade_id"].nunique()} atividades')
print(f'PCAP via justificativa: {just_pcap["atividade_id"].nunique()} atividades (novas)')

# ── 3. une as duas fontes ──────────────────────────────────────────────────
sol_pcap = pd.concat([sol_pcap, just_pcap], ignore_index=True).drop_duplicates(subset=['solicitacao_id'])

raw_pcap_df = sol_pcap.merge(pcap_props_df, on='pcap_num', how='inner')
print(f'raw_pcap_df:       {raw_pcap_df.shape}')
print(f'atividades unicas: {raw_pcap_df["atividade_id"].nunique()}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 29, Finished, Available, Finished, False)

PCAP via complemento:   2170 atividades
PCAP via justificativa: 1248 atividades (novas)
raw_pcap_df:       (3441, 24)
atividades unicas: 3341


### Detalhamento PCAP

In [28]:
# Detalhamento por grupo_item (texto para exibição ao usuário)
_det = (
    raw_pcac_det_df
    .groupby(['id_proposta', 'grupo_item'], as_index=False)['valor_total_item']
    .sum()
    .sort_values(['id_proposta', 'valor_total_item'], ascending=[True, False])
)

def _fmt_brl(v):
    return f'R$ {v:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

_det_grupos = (
    _det
    .groupby('id_proposta')
    .apply(lambda g: '\n'.join(
        f"{row['grupo_item']}: {_fmt_brl(row['valor_total_item'])}"
        for _, row in g.iterrows()
    ))
    .reset_index(name='det_grupos')
)

raw_pcap_df = raw_pcap_df.merge(_det_grupos, on='id_proposta', how='left')
print(f'raw_pcap_df com det_grupos: {raw_pcap_df.shape}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 30, Finished, Available, Finished, False)

raw_pcap_df com det_grupos: (3441, 25)


In [29]:
### Cria autonomia com todas as regras

autonomias_df = build_autonomias(rps_parcial_df, datas_df, contracts_df, raw_pcap_df)
autonomias_df['autonomia'].value_counts()

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 31, Finished, Available, Finished, False)

Via PCAP -> DIREG: 19 atividade(s)
Via PCAP -> STS:   156 atividade(s)


autonomia
UO       29728
STS       4050
DIREG     1639
Name: count, dtype: int64

## 13. Tabela Base


In [ ]:
def compute_capacidade(raw_acoes_df, datas_df, raw_datas_sessoes_df):
    """Campos de capacidade/público consolidados por atividade_id.

    Campos retornados:
    - capacidade_sessao  : lotação do espaço por sessão (a.lugares numérico; NaN se não preenchido)
    - estimativa_sessao  : estimativa de público por sessão (a.estimativa_publico numérico; NaN se não preenchido)
    - tipo_per_capita    : 0 = público acumula por sessão (Apresentação, Oficina etc.)
                           1 = público é total da atividade (Curso, Seminário)
    - capacidade_total   : capacidade_sessao × qt_sessoes (tipo=0) ou capacidade_sessao (tipo=1).
                           Quando capacidade_sessao > 4000 usa direto (evita inflação por sessão).
                           NaN se capacidade_sessao for nulo.
    - estimativa_total   : mesma lógica aplicada a estimativa_sessao.
    - capacidade_confiavel: True quando há ao menos um campo > 0 E a atividade não tem sessões
                            em locais abertos/externos (sem lotação física definível).
                            False indica que per_capita não deve ser calculado para esta atividade.

    Critério de capacidade_confiavel=False:
    - ambos campos nulos/zero: nenhuma informação de capacidade disponível.
    - tem_sessao_aberta: atividade tem ≥1 sessão em Área de Convivência, Praça, Fora da Unidade,
      Hall, Lobby etc. — locais sem lotação física definível.
      Nota: locais SEMIFIXOS (ginásio, quadra) NÃO tornam capacidade_confiavel=False;
      a variância de capacidade nesses locais é esperada (futebol vs. show).

    Para conferência no BI:
    - Filtrar capacidade_confiavel=False → cruzar com TipologiaLocal/localNome de datas_sessoes
      → deve concentrar em Área de Convivência, Fora da Unidade, Praça.
    - Per_capita correto (DAX): DIVIDE(SUM(custo_total), SUMX(FILTER(tabela_base,
      tabela_base[capacidade_confiavel]), tabela_base[capacidade_total]))
    """
    df = raw_acoes_df[['atividade_id', 'a.lugares', 'a.estimativa_publico', 'servico']].copy()
    df['capacidade_sessao'] = pd.to_numeric(df['a.lugares'],            errors='coerce')
    df['estimativa_sessao'] = pd.to_numeric(df['a.estimativa_publico'], errors='coerce')
    df['tipo_per_capita']   = df['servico'].isin(['Curso', 'Seminário']).astype(int)

    df = df.merge(
        datas_df[['atividade_id', 'qt_sessoes']].drop_duplicates(),
        on='atividade_id', how='left',
    )
    sessoes = pd.to_numeric(df['qt_sessoes'], errors='coerce').fillna(1)

    # capacidade_total / estimativa_total: acumula por sessão (tipo=0) ou usa direto (tipo=1)
    # Cap de 4000 para evitar inflação: atividades com lugares > 4000 usam valor direto
    for src_col, dst_col in [('capacidade_sessao', 'capacidade_total'),
                              ('estimativa_sessao',  'estimativa_total')]:
        v = df[src_col].astype(float)
        df[dst_col] = np.where(
            v.isna(), np.nan,
            np.where(v > 4000, v,
                np.where(df['tipo_per_capita'] == 0, sessoes * v, v)
            )
        )

    # capacidade_confiavel: False quando ambos nulos/zero OU tem sessão em local aberto/externo
    # Locais abertos: sem lotação física definível (área de convivência, praça, espaço externo etc.)
    # Locais SEMIFIXOS (ginásio, quadra) não são marcados: variância de capacidade é esperada
    _RE_ABERTO = (r'praça|convivência|externo|fora|parque|jardim|calçada|pátio'
                  r'|varanda|átrio|hall|foyer|lobby')
    _s = raw_datas_sessoes_df[['atividade_id', 'localNome', 'TipologiaLocal', 'TipoLocal']].copy()
    _s['eh_aberto'] = (
        _s['localNome'].fillna('').str.lower().str.contains(_RE_ABERTO)
        | _s['TipologiaLocal'].fillna('').str.lower().str.contains(_RE_ABERTO)
        | (_s['TipoLocal'] == 'externa')
    )
    _ativ_abertos = (
        _s.groupby('atividade_id')['eh_aberto'].any()
        .rename('tem_sessao_aberta').reset_index()
    )
    df = df.merge(_ativ_abertos, on='atividade_id', how='left')
    df['tem_sessao_aberta'] = df['tem_sessao_aberta'].fillna(False)

    _cap_zero = df['capacidade_sessao'].fillna(0) <= 0
    _est_zero = df['estimativa_sessao'].fillna(0)  <= 0
    df['capacidade_confiavel'] = ~((_cap_zero & _est_zero) | df['tem_sessao_aberta'])

    return df[['atividade_id', 'capacidade_sessao', 'estimativa_sessao',
               'tipo_per_capita', 'capacidade_total', 'estimativa_total',
               'capacidade_confiavel']]


In [30]:
SUBATIV_PERMANENTE = frozenset({
    'Acesso a recursos informacionais',
    'Análise de risco em saúde',
    'Consulta',
    'Exercício físicos sistematicos',
    'Formação esportiva',
    'Sessão diagnóstica/clínica',
    'Refeição',
    'Lanche',
    'Capacitação e Desenvolvimento de Empregados',
    'Rádio e TV',
    'Colônias recreativas',
    'Relacionamento com clientes',
    'Distribuição de doações',
    'Creche',
    'Pré-escola',
    'Acesso a recursos informacionais',
    'Procedimentos clínicos',
    'Procedimentos complementares',
    'Práticas coletivas',
    'Parque aquático',
    'Hospedagem',
})

SUBATIV_EVENTUAL = frozenset({
    'Apresentação',
    'Competições físico-esportivas',
    'Multipráticas recreativas',
    'Incentivo artístico e cultural',
    'Eventos',
    'Exibição',
    'Exposição',
    'Passeios',
    'Produtos gastronômicos',
    'Viagens',
})


def build_tabela_base(
    raw_acoes_df, datas_df, autonomias_df, raw_projetos_df,
    todas_as_datas_df, raw_tags_df, solicitacoes_desc_df, contracts_df,
    raw_acessibilidade_df, raw_pcap_df, raw_datas_sessoes_df,
    precif_df,
    justificativa_df,
) -> pd.DataFrame:

    _acoes_drop = ['projeto'] + [c for c in raw_acoes_df.columns if c == 'primeiradata' or c.endswith('.primeiradata')]
    df = raw_acoes_df.drop(columns=_acoes_drop, errors='ignore').copy()

    # ── linguagem artística (derivada de atividade) ──────────────────────────────────────
    _LINGUAGEM_POR_ATIVIDADE = {
        'Artes Cênicas - Dança':  'Dança',
        'Artes Cênicas - Circo':  'Circo',
        'Artes Cênicas - Teatro': 'Teatro',
        'Audiovisual':            'Audiovisual',
        'Biblioteca':             'Literatura',
        'Literatura':             'Literatura',
        'Música':                 'Música',
    }
    _ativ_col = 'atividade' if 'atividade' in df.columns else 'a.atividade'
    df['linguagem'] = df[_ativ_col].map(_LINGUAGEM_POR_ATIVIDADE).fillna('...')

    # ── faixa etária ──────────────────────────────────────────────────────
    df['a.idade_inicial'] = pd.to_numeric(df['a.idade_inicial'], errors='coerce').fillna(0)
    df['a.idade_final']   = pd.to_numeric(df['a.idade_final'],   errors='coerce').fillna(0)
    temp0 = df['a.idade_inicial'] + df['a.idade_final']
    temp1 = np.select(
        [df['a.linguagem'] == 'Crianças',
         df['a.linguagem'] == 'Idosos',
         df['a.recomendacao_etaria'] != 'Livre'],
        ['infantil', 'pessoas idosas', 'não é'],
        default='pode ser',
    )
    temp2 = np.select(
        [temp0 == 0, df['a.idade_inicial'] >= 60, df['a.idade_inicial'] >= 12,
         df['a.idade_final'] == 0, df['a.idade_final'] <= 13, df['a.idade_final'] <= 15],
        ['s/i', 'idosos', 'não é', 's/i', 'infantil', 'pode ser'],
        default='não é',
    )
    mesclado = pd.Series(np.array(temp1, dtype=object) + ' - ' + np.array(temp2, dtype=object), index=df.index)
    df['faixa'] = np.select(
        [mesclado.str.contains('infantil'), mesclado.str.contains('idosos'),
         mesclado.str.contains('pode ser'), mesclado.str.contains('s/i')],
        ['infantil', 'pessoas idosas', 's/i', 's/i'],
        default='s/i',
    )

    # ── joins 1:1 ─────────────────────────────────────────────────────────
    datas_cols = [
        'atividade_id', 'PrimeiraData', 'PrimeiraHora',
        'ultimadata', 'qt_sessoes', 'qt_datas_distintas', 'qt_horas',
        'tempo_da_sessao', 'diascorridos', 'mes',
        '30dias', '90dias', '60horas', 'ExtrapolaDataHora',
        'ano_2018', 'ano_2019', 'ano_2020', 'ano_2021', 'ano_2022',
        'ano_2023', 'ano_2024', 'ano_2025', 'ano_2026',
    ]
    df = df.merge(
        datas_df[[c for c in datas_cols if c in datas_df.columns]],
        on='atividade_id', how='left',
    )
    df = df.merge(
        autonomias_df[['atividade_id', 'autonomia', 'autonomiaTemporal', 'autonomiaCusto', 'autonomiaPCAP']],
        on='atividade_id', how='left',
    )
    df = df.merge(
        raw_projetos_df.drop(columns=['institucional'], errors='ignore'),
        on='projeto_id', how='left',
    )
    df = df.merge(todas_as_datas_df, on='atividade_id', how='left')
    df = df.merge(
        raw_tags_df.drop_duplicates('atividade_id')[['atividade_id', 'todas_as_tags']],
        on='atividade_id', how='left',
    )
    df = df.merge(solicitacoes_desc_df, on='atividade_id', how='left')
    for _f in ['tem_contrato', 'tem_passagem', 'tem_hospedagem', 'alerta']:
        if _f in df.columns:
            df[_f] = df[_f].fillna(0).astype(int)
    df = df.merge(
        contracts_df.drop_duplicates('atividade_id')[
            ['atividade_id', 'custo_contratos_total', 'custo_total', 'n_contratos', 'n_solic']
        ],
        on='atividade_id', how='left',
    )

    # capacidade/público consolidados (campos canônicos para per_capita no BI)
    cap_df = compute_capacidade(raw_acoes_df, datas_df, raw_datas_sessoes_df)
    df = df.merge(cap_df, on='atividade_id', how='left')

    df = df.merge(
        precif_df[['atividade_id', 'gratuito', 'maior_valor', 'menor_valor']],
        on='atividade_id', how='left',
    )
    _prec_col = 'precificacao_desc' if 'precificacao_desc' in df.columns else 'a.precificacao_desc'
    df['ingresso'] = np.where(df['gratuito'] == 'Sim', 'Gratuito', df[_prec_col])

    # ── periodicidade ─────────────────────────────────────────────────────
    # Regras (mutuamente exclusivas):
    # 1. Subatividades sempre permanentes
    # 2. Subatividades sempre eventuais (Ações formativas/mediadas excluem Curso/Vivência)
    # 3. Curso (subativ=Ações formativas) e Vivência (subativ=Ações mediadas):
    #    diascorridos > 90 E qt_sessoes > 30 → permanente; caso contrário → eventual
    dias = pd.to_numeric(df['diascorridos'], errors='coerce').fillna(0)
    sess = pd.to_numeric(df['qt_sessoes'],   errors='coerce').fillna(0)

    cond_perm  = df['subatividade'].isin(SUBATIV_PERMANENTE)
    cond_ev    = (
        df['subatividade'].isin(SUBATIV_EVENTUAL) |
        ((df['subatividade'] == 'Ações formativas') & (df['servico'] != 'Curso')) |
        ((df['subatividade'] == 'Ações mediadas')   & (df['servico'] != 'Vivência'))
    )
    cond_cv    = (
        ((df['subatividade'] == 'Ações formativas') & (df['servico'] == 'Curso')) |
        ((df['subatividade'] == 'Ações mediadas')   & (df['servico'] == 'Vivência'))
    )
    cv_perm = cond_cv & (dias > 90) & (sess > 30)
    cv_ev   = cond_cv & ~((dias > 90) & (sess > 30))

    df['periodicidade'] = np.select(
        [cond_perm, cond_ev, cv_perm, cv_ev],
        ['permanente', 'eventual', 'permanente', 'eventual'],
        default='s/i',
    )

    # Permanente sem contrato → autonomia UO (independente das outras fontes)
    _sem_contrato = df['custo_contratos_total'].fillna(0) == 0
    df.loc[(df['periodicidade'] == 'permanente') & _sem_contrato, 'autonomia'] = 'UO'

    # ── flags via .isin() ─────────────────────────────────────────────────
    df['tem_dispositivo'] = df['atividade_id'].isin(raw_acessibilidade_df['atividade_id']).astype(int)
    _acess_agg = (
        raw_acessibilidade_df[['atividade_id', 'a.identificacao']]
        .dropna(subset=['a.identificacao'])
        .groupby('atividade_id')
        .agg(
            todas_os_dispositivos=('a.identificacao', lambda x: ' | '.join(sorted(x.unique()))),
            n_dispositivos=('a.identificacao', 'nunique'),
        )
        .reset_index()
    )
    df = df.merge(_acess_agg, on='atividade_id', how='left')
    df['todas_os_dispositivos'] = df['todas_os_dispositivos'].fillna('')
    df['n_dispositivos'] = df['n_dispositivos'].fillna(0).astype(int)
    df['com_pcap']        = df['atividade_id'].isin(raw_pcap_df['atividade_id']).astype(int)

    # ── espaco_brincar: OR de quatro fontes ───────────────────────────────
    _eb = set().union(
        raw_acoes_df.loc[raw_acoes_df['a.nome'].str.contains('Espaço de Brincar', na=False), 'atividade_id'],
        raw_tags_df.loc[raw_tags_df['tag_nome'].str.contains('Espaço de Brincar', na=False), 'atividade_id'],
        raw_acoes_df.loc[raw_acoes_df['projeto_id'].isin(set(
            raw_projetos_df.loc[
                raw_projetos_df['projeto_uo_nome'].str.contains('Espaço de Brincar', na=False) |
                raw_projetos_df['tag_projeto'].str.contains('Espaço de Brincar', na=False),
                'projeto_id',
            ]
        )), 'atividade_id'],
        raw_datas_sessoes_df.loc[
            raw_datas_sessoes_df['localNome'].str.contains('Espaço de Brincar', na=False) |
            raw_datas_sessoes_df['TipologiaLocal'].str.contains('Espaço de Brincar', na=False),
            'atividade_id',
        ],
    )
    df['espaco_brincar'] = df['atividade_id'].isin(_eb).astype(int)

    # ── integra_expo: exposições longas (>30 sessões) com projeto → propaga a flag para todo o projeto
    _projetos_expo = set(
        df.loc[
            (df['servico'] == 'Exposição') &
            (sess > 30) &
            df['projeto_id'].notna() &
            (df['projeto_id'] != ''),
            'projeto_id',
        ]
    )
    df['integra_expo'] = (
        df['projeto_id'].isin(_projetos_expo) & df['projeto_id'].notna()
    ).astype(int)

    # ── periodicidade: refinamentos pós-flags ─────────────────────────────────
    # Vivência + espaco_brincar + sessoes > 18 → permanente
    _viv_eb = (
        (df['servico'] == 'Vivência')
        & (sess > 18)
        & (df['espaco_brincar'] == 1)
    )
    # Curso + sessoes > 18 + Curumim/Juventudes/Centro de Música em nome, projeto_nome ou complemento
    _RE_PERM_CURSO = r'curumim|juventudes?|centro de m[uú]sica'
    _nome_proj = (
        df['a.nome'].fillna('').str.contains(_RE_PERM_CURSO, case=False)
        | df['projeto_nome'].fillna('').str.contains(_RE_PERM_CURSO, case=False)
        | df['a.complemento'].fillna('').str.contains(_RE_PERM_CURSO, case=False)
    )
    _curso_prog = (df['servico'] == 'Curso') & (sess > 18) & _nome_proj
    df.loc[_viv_eb | _curso_prog, 'periodicidade'] = 'permanente'
    df.loc[(_viv_eb | _curso_prog) & _sem_contrato, 'autonomia'] = 'UO'

    # ── tem_educador: educadores no campo complemento ─────────────────────────
    # tem_educador=1: educador+contexto (ou agente ambiental) com custo_contratos_total=0
    # tem_educador=2: idem com custo_contratos_total>0
    _comp = df['a.complemento'].fillna('')
    _RE_CTX = (
        r'tecnologia[s]? e arte[s]?'
        r'|\bsesc\b'
        r'|\beta\b'
        r'|da exposi[cç][aã]o'
        r'|f[ií]sico[- ]esportiv[ao]s?'
        r'|infanto[- ]juveni[ls]'
        r'|curumim'
        r'|juventudes?'
        r'|espa[cç]o de brincar'
        r'|centro de m[uú]sica'
        r'|da unidade'
        r'|s[oó]cio[- ]?educativ[ao]s?'
        r'|agente de educa[cç][aã]o ambiental'
        r'|atividades? musicais?'
    )
    _has_educ_ctx = (
        _comp.str.contains(r'educador[ae]?s?', case=False)
        & _comp.str.contains(_RE_CTX, case=False)
    ) | _comp.str.contains(r'agente de educa[cç][aã]o ambiental', case=False)
    _custo_educ = pd.to_numeric(df['custo_contratos_total'], errors='coerce').fillna(0)
    df['tem_educador'] = np.where(
        _has_educ_ctx & (_custo_educ == 0), 1,
        np.where(_has_educ_ctx & (_custo_educ > 0), 2, 0)
    ).astype(int)

    # tem_educador=1 -> autonomia UO independente de dias corridos e carga horária
    df.loc[df['tem_educador'] == 1, 'autonomia'] = 'UO'

    # ── reordenação de colunas ────────────────────────────────────────────
    df = df.merge(justificativa_df[['atividade_id', 'justificativa']], on='atividade_id', how='left')

    col_order = [
        # Identificação
        'uo', 'atividade_id', 'status_atividade', 'nome', 'a.complemento',
        # Hierarquia programática
        'areaprog', 'atividade', 'subatividade', 'servico', 'periodicidade',
        # Classificação
        'tipo', 'subtipo', 'formato', 'linguagem',
        # Público e faixa etária
        'recomendacao_etaria', 'faixa',
        # Capacidade/público consolidados (canônicos para per_capita)
        'capacidade_sessao', 'estimativa_sessao', 'tipo_per_capita',
        'capacidade_total', 'estimativa_total', 'capacidade_confiavel',
        # Datas e sessões
        'PrimeiraData', 'PrimeiraHora', 'ultimadata',
        'qt_sessoes', 'qt_datas_distintas', 'qt_horas', 'tempo_da_sessao',
        'diascorridos', 'mes', '30dias', '90dias', '60horas', 'ExtrapolaDataHora',
        # Flags de ano
        'ano_2018', 'ano_2019', 'ano_2020', 'ano_2021', 'ano_2022',
        'ano_2023', 'ano_2024', 'ano_2025', 'ano_2026',
        # Texto de datas
        'todas_as_datas',
        # Autonomia
        'autonomia', 'autonomiaTemporal', 'autonomiaCusto', 'autonomiaPCAP',
        # Projeto
        'projeto_id', 'projeto_nome', 'projeto_complemento', 'projeto_categoria',
        'tag_projeto', 'tag_grupo_projeto', 'projeto_uo_nome',
        'projeto_descricao', 'projeto_comunicacao', 'projeto_conceitual', 'tem_pai',
        'justificativa',
        # Tags
        'todas_as_tags',
        # Custos — flags e totais
        'tem_contrato', 'tem_passagem', 'tem_hospedagem',
        'custo_contratos_total', 'custo_total', 'n_contratos', 'n_solic',
        # Custos — descrições
        'item_desc', 'contratos_desc', 'passagem_desc', 'hospedagem_desc',
        # Flags especiais
        'tem_dispositivo', 'todas_os_dispositivos', 'n_dispositivos', 'espaco_brincar', 'integra_expo', 'com_pcap', 'tem_educador',
        # Complementares
        'ingresso', 'gratuito', 'maior_valor', 'menor_valor', 'precificacao_desc', 'produtor', 'tem_parceria',
        'contatofornecedores', 'uso_interno', 'manutencao', 'regular',
        'integracao_sgc', 'institucional',
    ]
    ordered = [c for c in col_order if c in df.columns]
    extras  = [c for c in df.columns if c not in ordered]
    return df[ordered + extras]

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 32, Finished, Available, Finished, False)

In [31]:
# Justificativa: concatena textos de raw_acoes_txts separados por linha em branco
_txt_cols = ['sinopse_curta', 'sinopse_aprovacao', 'info_parceria', 'just_recursos']

justificativa_df = (
    raw_acoes_txts_df[['acao.atividade_id'] + [c for c in _txt_cols if c in raw_acoes_txts_df.columns]]
    .rename(columns={'acao.atividade_id': 'atividade_id'})
    .drop_duplicates(subset=['atividade_id'])
    .copy()
)

def _join_partes(row):
    parts = []
    for c in _txt_cols:
        if c in row.index:
            val = '' if pd.isna(row[c]) else str(row[c]).strip()
            if val:
                parts.append(val)
    return '\n\n'.join(parts)

justificativa_df['justificativa'] = justificativa_df.apply(_join_partes, axis=1)
justificativa_df = justificativa_df[['atividade_id', 'justificativa']]
print(f'justificativa_df: {justificativa_df.shape}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 33, Finished, Available, Finished, False)

justificativa_df: (21649, 2)


In [32]:
tabela_base_df = build_tabela_base(
    raw_acoes_df        = raw_acoes_df,
    datas_df            = datas_df,
    autonomias_df       = autonomias_df,
    raw_projetos_df     = raw_projetos_df,
    todas_as_datas_df   = todas_as_datas_df,
    raw_tags_df         = raw_tags_df,
    solicitacoes_desc_df= solicitacoes_desc_df,
    contracts_df        = contracts_df,
    raw_acessibilidade_df = raw_acessibilidade_df,
    raw_pcap_df         = raw_pcap_df,
    raw_datas_sessoes_df= raw_datas_sessoes_df,
    precif_df           = precif_df,
    justificativa_df    = justificativa_df,
)

print(f'tabela_base_df: {tabela_base_df.shape}')
print(f'atividade_id única: {tabela_base_df["atividade_id"].is_unique}')
print()
print('faixa:')
print(tabela_base_df['faixa'].value_counts())
print()
print('autonomia:')
print(tabela_base_df['autonomia'].value_counts())
print()
flags = ['tem_dispositivo', 'espaco_brincar', 'integra_expo', 'com_pcap',
         'tem_contrato', 'tem_passagem', 'tem_hospedagem']
for f in flags:
    if f in tabela_base_df.columns:
        print(f'{f}=1: {tabela_base_df[f].sum()}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 34, Finished, Available, Finished, False)

tabela_base_df: (35417, 91)
atividade_id única: True

faixa:
faixa
s/i               30528
infantil           3890
pessoas idosas      999
Name: count, dtype: int64

autonomia:
autonomia
UO       30731
STS       3765
DIREG      921
Name: count, dtype: int64

tem_dispositivo=1: 1767
espaco_brincar=1: 1090
integra_expo=1: 20971
com_pcap=1: 3341
tem_contrato=1: 15011.0
tem_passagem=1: 588.0
tem_hospedagem=1: 2982.0


In [ ]:
# ── bins para histogramas de custo (colunas físicas — Direct Lake não suporta calculadas)
_BINS_SESSAO = [0,   1_000,   2_500,   5_000, 10_000, 15_000, 20_000, 25_000 float('inf')]
_LBLS_SESSAO = ['1. 0–1k', '2. 1k–2.5k', '3. 2.5k–5k', '4. 5k–10k', '5. 10k–15k', '6. 15k–20k', '7. 20k-25k', '8. 25k+']

_BINS_HORA   = [0,    200,   400,   600,   800, 1_000, 1_500, 2_000, float('inf')]
_LBLS_HORA   = ['1. 0–200', '2. 200–400', '3. 400–600', '4. 600–800', '5. 800–1k', '6. 1k–1.5k', '7. 1.5k–2k', '8. 2k+']

_BINS_CAPITA = [0,    25,    50,    75,   100,   200,   400,  600, float('inf')]
_LBLS_CAPITA = ['1. 0–25', '2. 25–50', '3. 50–75', '4. 75–100', '5. 100–200', '6. 200–400', '7. 400–600', '8. 600+']


def _faixa(series, bins, labels):
    """pd.cut → string; NaN → 'n/a' (métrica não calculável para essa combinação)."""
    return pd.cut(series, bins=bins, labels=labels, right=True).astype('object').fillna('n/a')


def build_hist_custo(tabela_base_df, raw_datas_sessoes_df):
    """Tabelas pré-agregadas de custo para histogramas no Power BI.

    O Direct Lake não permite colunas calculadas: os bins (faixas de custo)
    precisam existir como colunas físicas no lakehouse.

    Dimensões principais: (uo, subatividade, servico, linguagem).
    Para Apresentação: tabela extra com (uo, tipologia_local) usando a tipologia
    predominante de datas_sessoes.

    Métricas computadas como razão de somas (não média de razões):
    - custo_por_sessao        : SUM(custo_total) / SUM(qt_sessoes)
    - custo_contr_por_sessao  : SUM(custo_contratos_total) / SUM(qt_sessoes)
    - custo_por_hora          : SUM(custo_total) / SUM(qt_horas)  [apenas SERVICOS_POR_HORA]
    - per_capita              : SUM(custo_total) / SUM(capacidade_total)  [capacidade_confiavel=True]

    Retorna (hist_custo_df, hist_apresentacao_df).
    """
    # ── filtro base ───────────────────────────────────────────────────────────
    _status = 'status_atividade' if 'status_atividade' in tabela_base_df.columns else 'a.status_atividade'
    if _status in tabela_base_df.columns:
        _b = tabela_base_df[tabela_base_df[_status].isin(['PENDENTE', 'APROVADO'])].copy()
    else:
        _b = tabela_base_df.copy()

    for col in ['custo_total', 'custo_contratos_total', 'qt_sessoes', 'qt_horas', 'capacidade_total']:
        _b[col] = pd.to_numeric(_b[col], errors='coerce').fillna(0)

    _b = _b[_b['custo_total'] > 0]

    # ── máscaras ──────────────────────────────────────────────────────────────
    # custo_por_hora: zeramos qt_horas onde o serviço não é faturado por hora
    _usa_hora  = _b['servico'].isin(SERVICOS_POR_HORA) | _b['subatividade'].isin(SUBATIV_POR_HORA)
    _confiavel = _b['capacidade_confiavel'] == True

    _b['_horas_hora'] = _b['qt_horas'].where(_usa_hora,  0)
    _b['_cap_conf']   = _b['capacidade_total'].where(_confiavel, 0)

    # ── agregação principal ────────────────────────────────────────────────────
    dims = ['uo', 'subatividade', 'servico', 'linguagem']

    hist = (
        _b.groupby(dims, as_index=False)
        .agg(
            n_atividades           = ('atividade_id',          'count'),
            total_custo            = ('custo_total',           'sum'),
            total_custo_contratos  = ('custo_contratos_total', 'sum'),
            total_sessoes          = ('qt_sessoes',            'sum'),
            total_horas_hora       = ('_horas_hora',           'sum'),
            total_capacidade_conf  = ('_cap_conf',             'sum'),
            n_com_capacidade_conf  = ('capacidade_confiavel',  'sum'),
        )
    )

    hist['custo_por_sessao']       = np.where(hist['total_sessoes']       > 0, hist['total_custo']           / hist['total_sessoes'],       np.nan)
    hist['custo_contr_por_sessao'] = np.where(hist['total_sessoes']       > 0, hist['total_custo_contratos'] / hist['total_sessoes'],       np.nan)
    hist['custo_por_hora']         = np.where(hist['total_horas_hora']    > 0, hist['total_custo']           / hist['total_horas_hora'],    np.nan)
    hist['per_capita']             = np.where(hist['total_capacidade_conf']> 0, hist['total_custo']          / hist['total_capacidade_conf'], np.nan)

    hist['faixa_custo_sessao']     = _faixa(hist['custo_por_sessao'],  _BINS_SESSAO, _LBLS_SESSAO)
    hist['faixa_custo_hora']       = _faixa(hist['custo_por_hora'],    _BINS_HORA,   _LBLS_HORA)
    hist['faixa_per_capita']       = _faixa(hist['per_capita'],        _BINS_CAPITA, _LBLS_CAPITA)

    # ── Apresentação × TipologiaLocal ─────────────────────────────────────────
    # Usa a tipologia de local mais frequente de cada atividade (modo)
    _apres_ids = _b.loc[_b['servico'] == 'Apresentação', 'atividade_id'].unique()

    _tip_modo = (
        raw_datas_sessoes_df[raw_datas_sessoes_df['atividade_id'].isin(_apres_ids)]
        [['atividade_id', 'TipologiaLocal']]
        .assign(TipologiaLocal=lambda d: d['TipologiaLocal'].fillna('Não informado'))
        .groupby('atividade_id')['TipologiaLocal']
        .agg(lambda x: x.mode().iloc[0])
        .rename('tipologia_local')
        .reset_index()
    )

    _apres = (
        _b[_b['servico'] == 'Apresentação']
        .merge(_tip_modo, on='atividade_id', how='left')
    )
    _apres['tipologia_local'] = _apres['tipologia_local'].fillna('Não informado')

    dims_ap = ['uo', 'tipologia_local']
    hist_apres = (
        _apres[_apres['qt_sessoes'] > 0]
        .groupby(dims_ap, as_index=False)
        .agg(
            n_atividades          = ('atividade_id',          'count'),
            total_custo           = ('custo_total',           'sum'),
            total_custo_contratos = ('custo_contratos_total', 'sum'),
            total_sessoes         = ('qt_sessoes',            'sum'),
        )
        .assign(
            custo_por_sessao      = lambda x: x['total_custo']           / x['total_sessoes'],
            custo_contr_por_sessao= lambda x: x['total_custo_contratos'] / x['total_sessoes'],
        )
    )
    hist_apres['faixa_custo_sessao'] = _faixa(hist_apres['custo_por_sessao'], _BINS_SESSAO, _LBLS_SESSAO)

    return hist, hist_apres


def add_faixas_base(df):
    """Adiciona faixa_* a tabela_base para histogramas com drill-through a atividade_id.

    Calcula por atividade (razao individual, nao razao de somas):
    - faixa_custo_sessao : custo_total / qt_sessoes
    - faixa_custo_hora   : custo_total / qt_horas  (apenas SERVICOS_POR_HORA / SUBATIV_POR_HORA)
    - faixa_per_capita   : custo_total / capacidade_total (apenas capacidade_confiavel=True)

    Preserva atividade_id para que um filtro num bin do histograma chegue
    as atividades individuais (drill-through no Power BI via tabela_base).
    """
    d = df.copy()
    _custo = pd.to_numeric(d['custo_total'],       errors='coerce').fillna(0)
    _sess  = pd.to_numeric(d['qt_sessoes'],        errors='coerce').fillna(0)
    _horas = pd.to_numeric(d['qt_horas'],          errors='coerce').fillna(0)
    _cap   = pd.to_numeric(d['capacidade_total'],  errors='coerce').fillna(0)

    _usa_hora  = d['servico'].isin(SERVICOS_POR_HORA) | d['subatividade'].isin(SUBATIV_POR_HORA)
    _confiavel = d['capacidade_confiavel'] == True

    custo_sessao = pd.Series(
        np.where(_sess  > 0, _custo / _sess,  np.nan), index=d.index)
    custo_hora   = pd.Series(
        np.where(_usa_hora & (_horas > 0), _custo / _horas, np.nan), index=d.index)
    per_capita   = pd.Series(
        np.where(_confiavel & (_cap  > 0), _custo / _cap,   np.nan), index=d.index)

    d['faixa_custo_sessao'] = _faixa(custo_sessao, _BINS_SESSAO, _LBLS_SESSAO)
    d['faixa_custo_hora']   = _faixa(custo_hora,   _BINS_HORA,   _LBLS_HORA)
    d['faixa_per_capita']   = _faixa(per_capita,   _BINS_CAPITA, _LBLS_CAPITA)
    return d


In [ ]:
hist_custo_df, hist_apresentacao_df = build_hist_custo(tabela_base_df, raw_datas_sessoes_df)

print(f'hist_custo_df:        {hist_custo_df.shape}')
print(f'hist_apresentacao_df: {hist_apresentacao_df.shape}')
print()
print('faixa_custo_sessao (hist_custo):')
print(hist_custo_df['faixa_custo_sessao'].value_counts().sort_index())
print()
print('faixa_per_capita (hist_custo):')
print(hist_custo_df['faixa_per_capita'].value_counts().sort_index())

# adiciona faixas a tabela_base para permitir drill-through a atividade_id
tabela_base_df = add_faixas_base(tabela_base_df)
print()
print(f'tabela_base_df com faixas: {tabela_base_df.shape}')
print('faixa_custo_sessao (tabela_base):')
print(tabela_base_df['faixa_custo_sessao'].value_counts().sort_index())
print()
print('faixa_per_capita (tabela_base):')
print(tabela_base_df['faixa_per_capita'].value_counts().sort_index())


### Verifica se os joins geraram mais de uma linha

In [33]:
# Isola as linhas duplicadas
raw_acoes_orig = read_raw(f'raw_acoes{_all}')
raw_acoes_orig['atividade_id'] = raw_acoes_orig['atividade_id'].astype(str).str.strip()

mask_dup = raw_acoes_orig.duplicated(subset=['atividade_id'], keep=False)
dupes = raw_acoes_orig[mask_dup].sort_values('atividade_id')

# Quais colunas variam entre as cópias de uma mesma atividade_id?
cols_variam = [
    col for col in dupes.columns
    if dupes.groupby('atividade_id')[col].nunique().max() > 1
]
print('Colunas com valores distintos entre duplicatas:')
print(cols_variam)

# Exemplo de uma atividade duplicada
exemplo = dupes['atividade_id'].iloc[0]
print(f'\nExemplo — atividade_id {exemplo}:')
dupes[dupes['atividade_id'] == exemplo][['atividade_id'] + cols_variam]


StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 35, Finished, Available, Finished, False)

Colunas com valores distintos entre duplicatas:
[]

Exemplo — atividade_id 44000000016317.0:


,atividade_id
30063,44000000016317.0
30062,44000000016317.0


## Define as gerências

In [34]:
# ── Gerências (tabela_base) ────────────────────────────────────────────────
#
# Três colunas derivadas:
#   gerencia  — gerência principal mapeada 1:1 de areaprog
#   gerenciaB — gerência secundária; regras baseadas em atividade / solicitações
#   gerencias — combinação pipe-sep das duas (sem pipe se igual; nulo se ambas nulas)
#
# ── Como adicionar novas regras em gerenciaB ──────────────────────────────
# 1. Crie um set/frozenset ou uma máscara booleana que identifique as atividades.
# 2. Adicione um bloco ".loc[mask, 'gerenciaB'] = 'SIGLA'" abaixo dos existentes,
#    ANTES das regras de prioridade mais alta (última linha escrita prevalece).
#    Exemplo: para nova regra com prioridade entre GESC e GEAVT, insira após GESC.
# ─────────────────────────────────────────────────────────────────────────

# ── gerencia ──────────────────────────────────────────────────────────────
GERENCIA_POR_AREAPROG = {
    'Gestão Cultural e Esportiva':        'GEDES-CPF',
    'Credenciamento':                     'GEARP',
    'Bem Viver':                          'GEDEP',
    'Conteúdo em Mídias':                 'GSD',
    'Eventos Físico-Esportivos':          'GDFE',
    'Programa de Ginástica Multifuncional': 'GDFE',
    'Programa de Práticas Aquáticas':     'GDFE',
    'Programa de Práticas Corporais':     'GDFE',
    'Programa Sesc de Esportes':          'GDFE',
    'Audiovisual':                        'GEAC',
    'Biblioteca':                         'GEAC',
    'Circo':                              'GEAC',
    'Dança':                              'GEAC',
    'Literatura':                         'GEAC',
    'Música':                             'GEAC',
    'Teatro':                             'GEAC',
    'Alimentação - Ações educativas':     'GEASA',
    'Sesc Mesa Brasil':                   'GEASA',
    'Alimentação':                        'GEASA',
    'Artes Visuais':                      'GEAVT',
    'Tecnologias e Artes':                'GEAVT',
    'Direitos humanos':                   'GEPROS',
    'Gênero e Sexualidade':               'GEPROS',
    'Infâncias':                          'GEPROS',
    'Juventudes':                         'GEPROS',
    'Negritude':                          'GEPROS',
    'Povos e Comunidades Tradicionais':   'GEPROS',
    'Povos Indígenas':                    'GEPROS',
    'Refúgio e Migração':                 'GEPROS',
    'Trabalho Social com Pessoas Idosas': 'GEPROS',
    'CEDEI':                              'GEPROS',
    'Desenvolvimento Comunitário':        'GESC',
    'Educação para Acessibilidade':       'GESC',
    'Educação para Sustentabilidade':     'GESC',
    'Turismo Social':                     'GESC',
    'Valorização Social':                 'GESC',
    'Qualidade de Vida':                  'GSO',
    'Saúde Bucal':                        'GSO',
    'Saúde Mental':                       'GSO',
    'Saúde Sexual e Reprodutiva':         'GSO',
}

tabela_base_df['gerencia'] = tabela_base_df['areaprog'].map(GERENCIA_POR_AREAPROG)

# ── gerenciaB ─────────────────────────────────────────────────────────────
# Regras avaliadas em ordem de prioridade crescente (última escrita prevalece).
# Prioridade: GEAC > GEAVT > GESC

# Regra 3 (prioridade baixa): solicitação com item_grupo='Acessibilidade' e custo > 0 → GESC
# Busca em raw_solicitacoes_df, que inclui todos os tipos de item (não só contratos).
ATIVIDADES_GERENB_GEAC = frozenset({
    'Artes Cênicas - Circo',
    'Artes Cênicas - Dança',
    'Artes Cênicas - Teatro',
    'Audiovisual',
    'Biblioteca',
    'Literatura',
    'Música',
})

# Regra 2: atividade = 'Artes Visuais' → GEAVT
ATIVIDADES_GERENB_GEAVT = frozenset({'Artes Visuais'})

_ids_acess_gesc = set(
    raw_solicitacoes_df.loc[
        (raw_solicitacoes_df['item_grupo'].astype(str).str.strip() == 'Acessibilidade') &
        (raw_solicitacoes_df['custo'] > 0),
        'atividade_id',
    ].unique()
)

_ativ = tabela_base_df['atividade'].fillna('').str.strip()

tabela_base_df['gerenciaB'] = None
# Aplica da prioridade mais baixa para a mais alta (última escrita vence):
tabela_base_df.loc[tabela_base_df['atividade_id'].isin(_ids_acess_gesc), 'gerenciaB'] = 'GESC'
tabela_base_df.loc[_ativ.isin(ATIVIDADES_GERENB_GEAVT), 'gerenciaB'] = 'GEAVT'
tabela_base_df.loc[_ativ.isin(ATIVIDADES_GERENB_GEAC),  'gerenciaB'] = 'GEAC'

# ── gerencias ─────────────────────────────────────────────────────────────
_g1 = tabela_base_df['gerencia'].fillna('')
_g2 = tabela_base_df['gerenciaB'].fillna('')

_so_g1   = (_g1 != '') & (_g2 == '')
_so_g2   = (_g1 == '') & (_g2 != '')
_iguais  = (_g1 != '') & (_g2 != '') & (_g1 == _g2)
_diferen = (_g1 != '') & (_g2 != '') & (_g1 != _g2)

tabela_base_df['gerencias'] = None
tabela_base_df.loc[_so_g1,   'gerencias'] = _g1[_so_g1]
tabela_base_df.loc[_so_g2,   'gerencias'] = _g2[_so_g2]
tabela_base_df.loc[_iguais,  'gerencias'] = _g1[_iguais]
tabela_base_df.loc[_diferen, 'gerencias'] = (_g1 + '|' + _g2)[_diferen]

# ── diagnóstico ───────────────────────────────────────────────────────────
print('gerencia:')
print(tabela_base_df['gerencia'].value_counts(dropna=False))
print()
print(f'gerenciaB (ids com Acessibilidade: {len(_ids_acess_gesc)}):')
print(tabela_base_df['gerenciaB'].value_counts(dropna=False))
print()
print('gerencias (top 15):')
print(tabela_base_df['gerencias'].value_counts(dropna=False).head(15))

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 36, Finished, Available, Finished, False)

gerencia:
gerencia
GEAC         9046
GDFE         6384
NaN          4982
GEPROS       4626
GEAVT        4354
GESC         2533
GEASA        1395
GSO          1134
GEDEP         560
GEDES-CPF     355
GEARP          36
GSD            12
Name: count, dtype: int64

gerenciaB (ids com Acessibilidade: 5):
gerenciaB
None     21475
GEAC      9618
GEAVT     4323
GESC         1
Name: count, dtype: int64

gerencias (top 15):
gerencias
GEAC            9081
GDFE            6369
None            4939
GEAVT           4358
GEPROS          4013
GESC            2461
GEASA           1393
GSO             1133
GEPROS|GEAC      599
GEDEP            560
GEDES-CPF        355
GESC|GEAC         65
GEARP             36
GEPROS|GEAVT      13
GDFE|GEAC         13
Name: count, dtype: int64


In [35]:
# ── Ponte gerência × ação (many-to-many para Power BI) ────────────────────
#
# dim_gerencia  : dimensão com as siglas únicas
# ponte_gerencia: bridge table atividade_id × sigla (uma linha por par)
#
# Permite filtrar tabela_base por gerência incluindo ações compartilhadas
# — onde a gerência do usuário aparece como principal OU secundária.

dim_gerencia_df = pd.DataFrame(
    {'sigla': sorted(set(GERENCIA_POR_AREAPROG.values()))}
)

ponte_gerencia_df = (
    tabela_base_df[['atividade_id', 'gerencias']]
    .dropna(subset=['gerencias'])
    .assign(sigla=lambda df: df['gerencias'].str.split('|'))
    .explode('sigla')
    .assign(sigla=lambda df: df['sigla'].str.strip())
    .loc[lambda df: df['sigla'] != '']
    [['atividade_id', 'sigla']]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f'dim_gerencia: {dim_gerencia_df.shape[0]} siglas')
print(f'ponte_gerencia: {ponte_gerencia_df.shape[0]} pares atividade x gerencia')
print(ponte_gerencia_df['sigla'].value_counts())

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 37, Finished, Available, Finished, False)

dim_gerencia: 11 siglas
ponte_gerencia: 31185 pares atividade x gerencia
sigla
GEAC         9764
GDFE         6384
GEPROS       4626
GEAVT        4385
GESC         2534
GEASA        1395
GSO          1134
GEDEP         560
GEDES-CPF     355
GEARP          36
GSD            12
Name: count, dtype: int64


## Limpa as colunas antes de salvar
- data e hora
- normaliza os nomes das colunas eliminando sujeiras antes do ponto
- transforma atividade_id e sessao_id em numérico decimal

In [36]:
import datetime

def sanitize_for_spark(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    # Remove prefixo de alias SQL (ex.: "a.complemento" → "complemento")
    df.columns = [col.split('.')[-1] for col in df.columns]

    df = df.loc[:, ~df.columns.duplicated()]

    # atividade_id e sessao_id como numérico decimal
    for id_col in ['atividade_id', 'sessao_id']:
        if id_col in df.columns:
            df[id_col] = pd.to_numeric(df[id_col], errors='coerce')

    for col in df.columns:

        # timedelta → segundos inteiros (DayTimeIntervalType não é suportado pelo Delta)
        if pd.api.types.is_timedelta64_dtype(df[col]):
            df[col] = df[col].dt.total_seconds().astype('Int64')
            continue

        amostra = df[col].dropna()
        if not len(amostra):
            continue
        sample = amostra.iloc[0]

        if isinstance(sample, datetime.time):
            # datetime.time → string; Spark/Arrow não suportam time64
            df[col] = df[col].apply(
                lambda t: t.strftime('%H:%M:%S') if isinstance(t, datetime.time) else None
            )
        elif isinstance(sample, (datetime.date, datetime.datetime)):
            df[col] = pd.to_datetime(df[col], errors='coerce')

    return df



def save_gold(df: pd.DataFrame, table_name: str) -> None:
    spark.createDataFrame(sanitize_for_spark(df)) \
         .write.mode('overwrite') \
         .option('overwriteSchema', 'true') \
         .saveAsTable(f'lake_gold_fatos.dbo.{table_name}')

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 38, Finished, Available, Finished, False)

### Transforma tags em áreas programáticas para filtragem expandida


In [37]:
# area_prog_tag: atividade_id + areaprogTag + origem

# -- Constantes de exclusão --
_EXCLUIR_GRUPO = {
    '', 'Abordagem (Projeto)', 'Acervo Sesc (Projeto)', 'Acao Online',
    'Bem Viver', 'Conteudos (Projeto)', 'Desenvolvimento de pessoal',
    'Editorial', 'Educacao formal', 'Familias', 'Formato (Projeto)',
    'Geral', 'Performance', 'Procedencia (Projeto)', 'Recorte Geografico (Projeto)',
    'Ação Online', 'Conteúdos (Projeto)', 'Famílias', 'Procedência (Projeto)',
    'Recorte Geográfico (Projeto)', 'Educação formal',
}
_EXCLUIR_FINAL = {
    'Caravana', 'Circuito Sesc', 'Circuito', 'Desenvolvimento de habilidades ',
    'destaques 2022', 'Destaques 2023', 'Destaques', 'Difusão de saberes',
    'Filosofia e Ciências Sociais', 'Mobilização - Plena Pausa', 'Mobilização',
    'Seminários', 'Sesc Gerações',
}

# -- Funções auxiliares --
def _calc_areaprog_tag(r):
    g, n = r['tag_grupo'], str(r['tag_nome'])
    if g == 'Diversidade Cultural': return n
    if g == 'Infâncias e Juventudes': return n
    if 'Mesa Brasil' in n: return 'Sesc Mesa Brasil'
    if g == 'Educação em Saúde': return n
    return g

def _calc_area_com_tag(r):
    n, g = str(r['tag_nome']), str(r['tag_grupo'])
    if n == 'Programa Sesc de Esportes': return 'Programa Sesc de Esportes'
    if 'Multifuncional' in n: return 'Programa de Ginástica Multifuncional'
    if 'Aquáticas' in n: return 'Programa de Atividades Aquáticas'
    if 'Corporais' in n: return 'Programa de Práticas Corporais'
    if 'Esporte' in g: return 'Eventos Físico-Esportivos'
    return 'areaprog_tag'

def _transform_tags(df):
    """Pipeline: df(atividade_id, tag_grupo, tag_nome) → df(atividade_id, areaprogTag)."""
    df = df[~df['tag_grupo'].fillna('').isin(_EXCLUIR_GRUPO)].copy()
    df['areaprog_tag'] = df.apply(_calc_areaprog_tag, axis=1)
    df['areaprog_tag'] = df['areaprog_tag'].str.replace('Do 13 ao 20', 'Negritude', regex=False)
    df = df[df['areaprog_tag'] != 'Diversidade Cultural']

    for _o, _n in [
        ('Crianças', 'Infâncias'), ('Bebês', 'Infâncias'), ('Espaço de Brincar', 'Infâncias'),
        ('Legítima Diferença', 'Gênero e Sexualidade'), ('Agosto Indígena', 'Povos Indígenas'),
        ('Jovens', 'Juventudes'), ('Adolescentes', 'Juventudes'), ('Abril Indígena', 'Povos Indígenas'),
        ('Culturas em Trânsito', 'Refúgio e Migração'), ('Semana Mundial do Brincar', 'Infâncias'),
        ('Coisa de Criança', 'Infâncias'),
    ]:
        df['areaprog_tag'] = df['areaprog_tag'].str.replace(_o, _n, regex=False)

    df['_act'] = df.apply(_calc_area_com_tag, axis=1)
    df['area_com_tag2'] = np.where(df['_act'] == 'areaprog_tag', df['areaprog_tag'], df['_act'])

    for _o, _n in [
        ('Educação em Saúde', 'Qualidade de Vida'), ('Nas Férias...', 'Infâncias'),
        ('Nutrição', 'Alimentação'), (' Urbanismo e Design', 'Tecnologias e Artes'),
        ('Arquitetura,Tecnologias e Artes', 'Tecnologias e Artes'),
        ('Audiovisual e Produção Sonora', 'Tecnologias e Artes'),
        (' Games e Cultura Geek', 'Tecnologias e Artes'), (' Eletrônica e Hardware', 'Tecnologias e Artes'),
        ('Ciência e Tecnologias', 'Tecnologias e Artes'), ('Sustentabilidade', 'Educação para Sustentabilidade'),
        ('Gestão Cultural', 'Gestão Cultural e Esportiva'), (' Costura e Moda', 'Tecnologias e Artes'),
        ('Curumim35anos', 'Infâncias'), (' Artesanato e Craft', 'Tecnologias e Artes'),
    ]:
        df['area_com_tag2'] = df['area_com_tag2'].str.replace(_o, _n, regex=False)

    for _o, _n in [
        ('Educação em Saúde', 'Qualidade de Vida'), ('Saúde', 'Qualidade de Vida'),
        ('Atividades formativas em gestão e mediação culturais', 'Gestão Cultural e Esportiva'),
        ('Acompanhamento GDFE', 'Eventos Físico-Esportivos'),
        ('Letramento e Inclusão Digital', 'Tecnologias e Artes'),
    ]:
        df['tag_nome'] = df['tag_nome'].astype(str).str.replace(_o, _n, regex=False)

    df['areaprogTag'] = np.where(df['area_com_tag2'] == 'null', df['tag_nome'], df['area_com_tag2'])
    df = df[['atividade_id', 'areaprogTag']].drop_duplicates()
    df = df[df['areaprogTag'].notna() & (df['areaprogTag'] != 'null') & (df['areaprogTag'].str.strip() != '')]

    for _o, _n in [
        ('Acessibilidade', 'Educação para Acessibilidade'),
        ('Cursos e Oficinas Artes Visuais', 'Tecnologias e Artes'),
        ('Curumim', 'Infâncias'),
        ('Educação para Educação para Sustentabilidade', 'Educação para Sustentabilidade'),
        ('Qualidade de vida', 'Qualidade de Vida'),
        ('Artesanias e ofícios tradicionais', 'Tecnologias e Artes'),
        ('Acompanhamento GDFE', 'Eventos Físico-Esportivos'),
        ('Trabalho Social com Pessoas Idosas', 'Pessoas Idosas'),
        ('Esporte e Atividade Física', 'Eventos Físico-Esportivos'),
        ('MPB', 'Música'),
        ('Arte têxtil e moda', 'Tecnologias e Artes'),
        ('Saúde mental e emocional [BV]', 'Bem Viver'),
        ('Audiovisual/Imagem e som', 'Audiovisual'),
        ('Cinema e audiovisual', 'Audiovisual'),
        ('Documentário', 'Audiovisual'),
        ('Cinema', 'Audiovisual'),
        ('Adulto - Ator', 'Teatro'),
        ('Adulto - Ator', 'Teatro'),
        ('Artes Visuais e gráficas', 'Artes Visuais'),
    ]:
        df['areaprogTag'] = df['areaprogTag'].str.replace(_o, _n, regex=False)

    df = df[~df['areaprogTag'].isin(_EXCLUIR_FINAL)]

    for _o, _n in [('PICS', 'Qualidade de Vida'), ('Gordofobia', 'Qualidade de Vida')]:
        df['areaprogTag'] = df['areaprogTag'].str.replace(_o, _n, regex=False)

    return df

# -- Parte 1: areaprog direto --
_ap_col = 'areaprog' if 'areaprog' in raw_acoes_df.columns else 'a.areaprog'
_pt1 = (
    raw_acoes_df[['atividade_id', _ap_col]]
    .rename(columns={_ap_col: 'areaprogTag'})
    .dropna(subset=['areaprogTag'])
    .copy()
)
_pt1 = _pt1[_pt1['areaprogTag'].str.strip().ne('') & _pt1['areaprogTag'].ne('null')]
_pt1['origem'] = 'areaprog'
_pt1 = _pt1.drop_duplicates(subset=['atividade_id', 'areaprogTag'])

# -- Parte 2: tags (raw_tags_df, já explodido) --
_t = raw_tags_df[['atividade_id', 'tag_nome', 'tag_grupo']].copy()
_t = _t[_t['atividade_id'].isin(raw_acoes_df['atividade_id'])]
_t = _transform_tags(_t)
_t['origem'] = 'tag'

# -- Parte 3: tags de projeto (pipe-separated em raw_projetos_df, join por projeto_id) --
_gp_col = next((c for c in raw_projetos_df.columns if 'tag_grupo_projeto' in c), None)
_tp_col = next((c for c in raw_projetos_df.columns if c == 'tag_projeto' or c.endswith('.tag_projeto')), None)
_pid_col = 'projeto_id'

if _gp_col and _tp_col and _pid_col in raw_acoes_df.columns:
    _proj_base = (
        raw_acoes_df[['atividade_id', _pid_col]]
        .dropna(subset=[_pid_col])
        .merge(raw_projetos_df[[_pid_col, _gp_col, _tp_col]], on=_pid_col, how='left')
        .dropna(subset=[_gp_col])
    )
    _rows = []
    for _, row in _proj_base.iterrows():
        grupos = [g.strip() for g in str(row[_gp_col]).split('|')] if pd.notna(row[_gp_col]) else []
        nomes  = [n.strip() for n in str(row[_tp_col]).split('|')] if pd.notna(row[_tp_col]) else []
        n_items = max(len(grupos), len(nomes))
        grupos += [''] * (n_items - len(grupos))
        nomes  += [''] * (n_items - len(nomes))
        _rows.extend(
            {'atividade_id': row['atividade_id'], 'tag_grupo': g, 'tag_nome': n}
            for g, n in zip(grupos, nomes)
        )
    _p = pd.DataFrame(_rows)
    _p = _transform_tags(_p)
    _p['origem'] = 'projeto'
else:
    _p = pd.DataFrame(columns=['atividade_id', 'areaprogTag', 'origem'])


# -- Combinar: prioridade areaprog > tag > projeto --
_priority = {'areaprog': 0, 'tag': 1, 'projeto': 2}
area_prog_tag_df = (
    pd.concat(
        [_pt1[['atividade_id', 'origem', 'areaprogTag']],
         _t[['atividade_id', 'origem', 'areaprogTag']],
         _p[['atividade_id', 'origem', 'areaprogTag']]],
        ignore_index=True
    )
    .sort_values('origem', key=lambda s: s.map(_priority))
    .drop_duplicates(subset=['atividade_id', 'areaprogTag'], keep='first')
    .reset_index(drop=True)
)

# -- Limpeza e normalização pós-concat --
for _o, _n in [('Trabalho Social com Pessoas Idosas', 'Pessoas Idosas')]:
    area_prog_tag_df['areaprogTag'] = area_prog_tag_df['areaprogTag'].str.replace(_o, _n, regex=False)
area_prog_tag_df = (
    area_prog_tag_df[
        area_prog_tag_df['areaprogTag'].notna() &
        ~area_prog_tag_df['areaprogTag'].isin({'null', 'nan', 'None'}) &
        area_prog_tag_df['areaprogTag'].str.strip().ne('')
    ]
    .drop_duplicates(subset=['atividade_id', 'areaprogTag'])
    .reset_index(drop=True)
)

print(f'area_prog_tag_df: {area_prog_tag_df.shape}')
print(f'atividades unicas: {area_prog_tag_df["atividade_id"].nunique()}')
print(f'areaprogTag unicos: {area_prog_tag_df["areaprogTag"].nunique()}')
print(area_prog_tag_df['origem'].value_counts())

StatementMeta(, 4179eda9-4d8e-4c66-8606-1c5e4a52f795, 39, Finished, Available, Finished, False)

area_prog_tag_df: (43252, 3)
atividades unicas: 30638
areaprogTag unicos: 46
origem
areaprog    30435
tag         10412
projeto      2405
Name: count, dtype: int64


In [ ]:
# ── Máscara de exibição para áreas programáticas ─────────────────────────────────────────
#
# Renomeia os valores de areaprog (base) e areaprogTag (area_prog_tag)
# antes de salvar. Edite os valores à direita para simplificar os nomes.
# Chaves não listadas ficam com o nome original.

AREAPROG_DISPLAY = {
    'Alimentação - Ações educativas':     'Alimentação Educ.',
    'Artes Cênicas - Dança':              'Dança',
    'Desenvolvimento Comunitário':        'Desenvolvimento Comunitário',
    'Direitos humanos':                   'Direitos humanos',
    'Educação para Acessibilidade':       'Acessibilidade',
    'Educação para Sustentabilidade':     'Sustentabilidade',
    'Eventos Físico-Esportivos':          'Eventos Físico-Esportivos',
    'Gênero e Sexualidade':               'Gênero e Sexualidade',
    'Gestão Cultural e Esportiva':        'Gestão Cultural e Esportiva',
    'Infâncias':                          'Infâncias',
    'Juventudes':                         'Juventudes',
    'Literatura':                         'Literatura',
    'Música':                             'Música',
    'Negritude':                          'Negritude',
    'Povos e Comunidades Tradicionais':   'Povos e Comunidades Tradicionais',
    'Povos Indígenas':                    'Povos Indígenas',
    'Programa de Ginástica Multifuncional': 'GMF',
    'Programa de Práticas Aquáticas':     'Práticas Aquáticas',
    'Programa de Práticas Corporais':     'Práticas Corporais',
    'Programa Sesc de Esportes':          'Programa Sesc de Esportes',
    'Qualidade de Vida':                  'Qualidade de Vida',
    'Refúgio e Migração':                 'Refúgio e Migração',
    'Saúde Bucal':                        'Saúde Bucal',
    'Saúde Mental':                       'Saúde Mental',
    'Saúde Sexual e Reprodutiva':         'Saúde Sexual e Reprodutiva',
    'Sesc Mesa Brasil':                   'Sesc Mesa Brasil',
    'Artes Cênicas - Teatro':             'Teatro',
    'Artes Cênicas - Circo':             'Circo',
    'Tecnologias e Artes':                'Tecnologias e Artes',
    'Trabalho Social com Pessoas Idosas': 'Pessoas Idosas',
    'Turismo Social':                     'Turismo Social',
    'Valorização Social':                 'Valorização Social',
}

tabela_base_df['areaprog'] = tabela_base_df['areaprog'].map(
    lambda v: AREAPROG_DISPLAY.get(v, v) if v is not None else v
)

area_prog_tag_df['areaprogTag'] = area_prog_tag_df['areaprogTag'].map(
    lambda v: AREAPROG_DISPLAY.get(v, v) if v is not None else v
)

print('Mascara AREAPROG_DISPLAY aplicada.')
print(f'areaprog unicos em base: {tabela_base_df["areaprog"].nunique()}')
print(f'areaprogTag unicos em area_prog_tag: {area_prog_tag_df["areaprogTag"].nunique()}')


## 14. Salvar em lake_gold_siplan


In [ ]:
# corrigir os nulos na hora em que aparecem: tem na tabela base mas não na de sessoes
tabela_base_df = tabela_base_df[tabela_base_df['PrimeiraData'].notna()]
print(f'tabela_base_df (após filtro sem PrimeiraData): {tabela_base_df.shape}')

save_gold(tabela_base_df,       f'base{_all}')
save_gold(raw_datas_sessoes_df, f'datas_sessoes{_all}')
save_gold(solicitacoes_df,      f'solicitacoes{_all}')
save_gold(hist_custo_df,        f'hist_custo{_all}')
save_gold(hist_apresentacao_df, f'hist_apresentacao{_all}')
save_gold(dim_gerencia_df,   'dim_gerencia')
save_gold(ponte_gerencia_df, f'ponte_gerencia{_all}')
save_gold(raw_pcap_df,          f'pcap{_all}')
acessibilidade_df = raw_acessibilidade_df[['atividade_id', 'a.identificacao']].copy()
save_gold(acessibilidade_df,    f'acessibilidade{_all}')
save_gold(raw_parcelas_df,      'parcelas')
save_gold(area_prog_tag_df,     f'area_prog_tag{_all}')
print('Salvo em lake_gold_fatos: base, datas_sessoes, solicitacoes, hist_custo, hist_apresentacao, pcap, acessibilidade, parcelas, area_prog_tag')